# Projet Caroline DEVRED : Analyse du marché automobile au USA

## 1. Chargement des données et découverte de ces dernières



*  Importer  les bibliothèques numpy et pandas, ainsi que les autres bibliothèques utilies à l'analyse.




In [277]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import re
from datetime import datetime
import pytz
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy.stats import pearsonr


In [278]:
# liens du fichier qui contient les données
link='/content/drive/MyDrive/car_prices.csv'

Ce que l'on sait sur les données :


*    year: Année de vente du véhicule

*    make: La marque du véhicule

*    model: Le modèle du véhicule

*    trim: Version spécifique du modèle spécifique du véhicule

*    body: Type de carrosserie du véhicule (e.g. SUV)

*    transmission: Type de transmission

*    vin: Numéro de chassis du véhicule, c'est un identifiant unique!

*    state: État dans lequel la vente a été réalisée

*    condition: Une évaluation de l'état du véhicule

*    odometer: Le nombre de kilométres ou de milles (?) parcourus par le véhicule

*    color: La couleur du véhicule

*    interior: La couleur intérieure du véhicule

*    seller: Identifiant de l'entité qui vend le véhicule

*    mmr: La côte ou prix de marché du véhicule

*    sellingprice: Le prix de vente final du véhicule

*    saledate: La date de vente du véhicule





In [279]:
# Chargement des données dans un dataframe nommé data

data =pd.read_csv(link,                         # chemin du fichier
                           sep = ',',                    # caractère séparant les valeurs
                           header = 0                   # numéro de la ligne contenant le nom des colonnes
                           )
print(data.shape)

data.head()

(558837, 16)


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.00,16639.00,white,black,kia motors america inc,20500.00,21500.00,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.00,9393.00,white,beige,kia motors america inc,20800.00,21500.00,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.00,1331.00,gray,black,financial services remarketing (lease),31900.00,30000.00,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.00,14282.00,white,black,volvo na rep/world omni,27500.00,27750.00,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.00,2641.00,gray,black,financial services remarketing (lease),66000.00,67000.00,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


## 2. Nettoyages des données

### 2.1 Gestion des string
Je decide de les mettre en minuscule sauf la date, que je gèrerai plus tard

In [280]:
# Colonnes à ne pas transformer (saledate)
exclude_cols = ['saledate']

for col in data.columns:
    if data[col].dtype == "object" and col not in exclude_cols:
        data[col] = data[col].str.lower()


### 2.2 Gestion des données dupliqées

* Y a-t-il des données duppliquées ?


In [281]:
print('nombre de ligne en double ?', data.duplicated().sum())

nombre de ligne en double ? 0


Pas de ligne duppliquée


### 2.3 Gestion de vin

* Gestion de vin. Est-il unique ? Est-il toujours présent ?

In [282]:
print('Nombre de voiture sans vin', data['vin'].isna().sum(), ' soit ',  data['vin'].isna().sum()/len(data) * 100, ' %' )
print('Nombre de duplicat de vin', data['vin'].duplicated().sum(), ' soit ',  data['vin'].duplicated().sum()/len(data) * 100, ' %' )

Nombre de voiture sans vin 4  soit  0.000715772219806491  %
Nombre de duplicat de vin 8539  soit  1.5279947462319068  %


  * affichage des vin manquants









In [283]:
data[data['vin'].isna()]

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
461612,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,NaN,3vwd17aj3fm259017,NaN,46.00,2711,white,black,NaN,14250.00,14000
505299,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,NaN,3vwd17aj7fm222388,NaN,36.00,20379,silver,black,NaN,13600.00,13500
529009,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,NaN,3vwd17aj8fm298895,NaN,2.00,2817,red,black,NaN,13750.00,12200
551222,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,NaN,3vwd17aj8fm239622,NaN,2.00,9562,silver,black,NaN,13200.00,12100


Les vin manquant ayant d'autres données problématiques (l'etat où sont les véhicule, leur condition) et representant un faible pourcentage du data set ( 0.0007), je vais les supprimer

In [284]:
data = data[~data['vin'].isna()].copy()
print('vin manquants :', data['vin'].isna().sum())

vin manquants : 0


* affichage des vin dupliqué, alors que le vin est sensé être unique

In [285]:
data[data['vin'].duplicated()]

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
7231,2005,chevrolet,equinox,ls,suv,automatic,2cndl13f056137366,ca,NaN,125141.00,red,—,buena park honda,3550.00,3300.00,Thu Dec 18 2014 12:00:00 GMT-0800 (PST)
7463,2003,bmw,7 series,745li,sedan,automatic,wbagn63403ds43612,ca,24.00,212596.00,white,black,prestige auto wholesale inc,1675.00,3300.00,Thu Dec 18 2014 12:00:00 GMT-0800 (PST)
18120,2007,gmc,envoy,denali,suv,automatic,1gket63m672242776,va,35.00,119475.00,gray,gray,blue knob auto sales inc,7725.00,7400.00,Thu Dec 18 2014 09:05:00 GMT-0800 (PST)
19043,2007,nissan,pathfinder,se,suv,NaN,5n1ar18w77c615027,nj,NaN,1.00,gray,—,awn sales ltd,10600.00,2600.00,Wed Dec 17 2014 09:30:00 GMT-0800 (PST)
23612,2004,acura,mdx,base,suv,NaN,2hnyd182x4h516719,il,NaN,1.00,NaN,NaN,silver auto sales inc,7100.00,1500.00,Thu Dec 18 2014 10:00:00 GMT-0800 (PST)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
558737,2009,jeep,wrangler,unlimited rubicon,suv,automatic,1j4ga69109l752295,nv,37.00,70423.00,white,black,millennium cars,23100.00,17500.00,Fri Jun 19 2015 05:00:00 GMT-0700 (PDT)
558738,2008,ford,f-250 super duty,lariat,crew cab,automatic,1ftsw21rx8ea22277,nv,33.00,125628.00,black,beige,bul connections llc,21000.00,18700.00,Fri Jun 19 2015 04:45:00 GMT-0700 (PDT)
558750,2007,saturn,aura,xe,sedan,automatic,1g8zs57n17f246542,ga,29.00,82083.00,—,beige,carworks inc,5450.00,5200.00,Tue Jun 23 2015 06:00:00 GMT-0700 (PDT)
558800,2012,kia,soul,base,wagon,manual,kndjt2a57c7424577,nv,28.00,53607.00,silver,black,unique autos,7825.00,8000.00,Fri Jul 03 2015 09:00:00 GMT-0700 (PDT)


Extraction et groupage des vin duppliqué

In [286]:
dup_vin = data[data['vin'].duplicated(keep=False)]
dup_vin.sort_values('vin')

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
49021,2000,acura,tl,3.2,sedan,automatic,19uua5663ya022038,fl,19.00,105420.00,gold,beige,autonation toyota scion weston,2150.00,1100.00,Tue Dec 23 2014 12:15:00 GMT-0800 (PST)
218119,2000,acura,tl,3.2,sedan,automatic,19uua5663ya022038,fl,19.00,105431.00,gold,tan,autonation toyota scion weston,2325.00,1000.00,Tue Jan 27 2015 10:00:00 GMT-0800 (PST)
160618,2006,acura,tl,base,sedan,manual,19uua65596a059705,nj,26.00,89661.00,white,brown,santander consumer,9025.00,8200.00,Wed Jan 28 2015 01:30:00 GMT-0800 (PST)
344335,2006,acura,tl,base,sedan,manual,19uua65596a059705,nj,25.00,89741.00,white,black,meridian remarketing,9100.00,8500.00,Wed Mar 04 2015 01:30:00 GMT-0800 (PST)
136582,2005,acura,tl,3.2,sedan,automatic,19uua66215a070166,ca,37.00,131725.00,silver,gray,premium auto wholesale,6400.00,7800.00,Thu Jan 15 2015 04:00:00 GMT-0800 (PST)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487782,2007,maserati,quattroporte,executive gt duoselect,sedan,automatic,zamce39a470026893,ca,34.00,46128.00,gray,gray,rose auto sales inc,26800.00,23500.00,Thu Jun 04 2015 05:30:00 GMT-0700 (PDT)
66731,2007,maserati,quattroporte,NaN,NaN,automatic,zamfe39a970030502,ga,35.00,27851.00,black,tan,jim ellis,28700.00,29500.00,Tue Dec 30 2014 12:30:00 GMT-0800 (PST)
252839,2007,NaN,NaN,NaN,NaN,automatic,zamfe39a970030502,ga,34.00,28000.00,black,beige,auto connection llc,29000.00,31000.00,Wed Feb 04 2015 02:00:00 GMT-0800 (PST)
415070,2014,fiat,500l,easy,wagon,automatic,zfbcfabh4ez025834,fl,47.00,13513.00,red,black,datura auto sales and rentals,12300.00,11500.00,Tue Jun 02 2015 02:15:00 GMT-0700 (PDT)


 Les dupplications sont très proches, on peut penser à la revente d'une même voiture

Je fais faire un test de cohérence avec le nombre de valeurs différentes par VIN

In [287]:
dup_vin.groupby('vin', as_index=False).agg(
    nb_lignes=('vin', 'size'),
    nb_valeurs_diff_sellingprice=('sellingprice', 'nunique'),
    nb_valeurs_diff_odometer=('odometer', 'nunique'),
    nb_valeurs_diff_condition=('condition', 'nunique'),
    nb_valeurs_diff_saledate=('saledate', 'nunique'),
).sort_values(by='nb_lignes', ascending=False)

,vin,nb_lignes,nb_valeurs_diff_sellingprice,nb_valeurs_diff_odometer,nb_valeurs_diff_condition,nb_valeurs_diff_saledate
6135,automatic,22,14,11,0,18
7656,wbanv13588cz57827,5,4,5,4,5
5703,5n1ar1nn2bc632869,4,4,4,4,4
6005,5uxfe43579l274932,4,3,4,3,4
7373,trusc28n241022003,4,3,4,4,4
...,...,...,...,...,...,...
2803,1n4aa5ap0bc862281,2,2,2,2,2
2804,1n4aa5ap0bc867237,2,2,2,2,2
2805,1n4aa5ap0cc810151,2,2,2,2,2
2806,1n4aa5ap0cc846244,2,2,2,2,2


Le vin automatic est une erreur. Que recouvre-t-il ?

In [288]:
data[data['vin'] == 'automatic']

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
408161,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,automatic,3vwd17aj4fm201708,NaN,46.00,4802,silver,gray,NaN,13200.00,16500
417835,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,automatic,3vwd17aj2fm258506,NaN,1.00,9410,white,gray,NaN,13300.00,10500
421289,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,automatic,3vwd17aj3fm276741,NaN,46.00,1167,blue,black,NaN,13200.00,12700
424161,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,automatic,3vwd17aj2fm285365,NaN,1.00,2172,gray,black,NaN,14050.00,8250
427040,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,automatic,3vwd17aj0fm227318,NaN,41.00,14872,gray,black,NaN,13700.00,14300
427043,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,automatic,3vwd17aj6fm218641,NaN,49.00,12655,red,black,NaN,13850.00,14500
434424,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,automatic,3vwd17aj7fm223475,NaN,46.00,15719,blue,black,NaN,13650.00,13500
444501,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,automatic,3vwd17aj5fm297123,NaN,2.00,6388,white,black,NaN,13850.00,10700
453794,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,automatic,3vwd17aj5fm219943,NaN,44.00,16633,silver,black,NaN,13600.00,13600
461597,2015,volkswagen,jetta,se pzev w/connectivity,navitgation,sedan,automatic,3vwd17aj9fm219766,NaN,44.00,11034,black,black,NaN,13900.00,13000


In [289]:
print('Nombre de voiture avec vin automatic vin', len(data[data['vin'] == 'automatic']), ' soit ',  len(data[data['vin'] == 'automatic'])/len(data) * 100, ' %' )

Nombre de voiture avec vin automatic vin 22  soit  0.003936775387280279  %


il y un décalage de **plusieurs** colonnes. Cela me semble trop difficile à corriger de manière fiable. Vu le faible pourcentage que cela represente, je vais donc les supprimer.

In [290]:
data = data[data['vin']!='automatic'].copy()
print(len(data[data['vin'] == 'automatic']))

0


Je m'occuppe des autres vin problématique
pour considérer que c’est la même voiture, il faut au minimum que make, model, trim et body soient identiques. Si l’un de ces champs diffère, je considère le VIN comme suspect plutôt que comme une revente certaine et je le supprime. S'il est nul, je ne le gère pas pour le moment

In [291]:
dup_vin = data[data['vin'].duplicated(keep=False)]
resume = dup_vin.groupby(level=0).agg(
    n_make=('make', 'nunique'),
    n_model=('model', 'nunique'),
    n_trim=('trim', 'nunique'),
    n_body=('body', 'nunique')
)
print(resume)

        n_make  n_model  n_trim  n_body
21           1        1       1       1
70           1        1       1       1
382          1        1       1       1
394          1        1       1       1
432          1        1       1       1
...        ...      ...     ...     ...
558737       1        1       1       1
558738       1        1       1       1
558750       1        1       1       1
558800       1        1       1       1
558808       1        1       1       1

[16841 rows x 4 columns]


In [292]:
vin_suspects = resume[(resume['n_make'] > 1) |
                      (resume['n_model'] > 1) |
                      (resume['n_trim'] > 1) |
                      (resume['n_body'] > 1)].index

data = data.drop(index=vin_suspects).copy()

In [293]:
# les noms de colonnes sont normalisée
# je jette un oeil sur les données
data.describe()
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 558811 entries, 0 to 558836
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   year          558811 non-null  int64  
 1   make          548510 non-null  object 
 2   model         548412 non-null  object 
 3   trim          548160 non-null  object 
 4   body          545616 non-null  object 
 5   transmission  493459 non-null  object 
 6   vin           558811 non-null  object 
 7   state         558811 non-null  object 
 8   condition     547017 non-null  float64
 9   odometer      558717 non-null  float64
 10  color         558062 non-null  object 
 11  interior      558062 non-null  object 
 12  seller        558811 non-null  object 
 13  mmr           558799 non-null  float64
 14  sellingprice  558799 non-null  float64
 15  saledate      558799 non-null  object 
dtypes: float64(4), int64(1), object(11)
memory usage: 72.5+ MB


On constate :
* Les données qualitatives possèdent les bons types
* la date va être à formatiser autrement
* il va falloir comment fonctionne condition
* il y a des données manquantes

### 2.4 Etude des données manquantes
 * Combien de données manquantes par colonne

In [294]:
data.isna().sum()

,0
year,0
make,10301
model,10399
trim,10651
body,13195
transmission,65352
vin,0
state,0
condition,11794
odometer,94


*  saledate et sellingprice
si les 12 lignes sont exactement les mêmes lignes où sellingprice et saledate manquent. je considèrerais que ce sont des ventes qui n'ont pas eu lieu et les supprimerai

In [295]:
mask = data['sellingprice'].isna() & data['saledate'].isna()
data[mask].shape

(12, 16)

In [296]:
# je supprime
data = data[~mask].copy()

In [297]:
data.isna().sum()

,0
year,0
make,10301
model,10399
trim,10651
body,13195
transmission,65351
vin,0
state,0
condition,11794
odometer,94


Statégie de selection des champs possédant des valeurs NaN à traiter : make, model, trim, body, transmission servent à segmenter le marché, mais ce ne sont pas toujours indispensables pour l’analyse globale du marché ; leur absence gêne surtout les analyses fines. Je laisse en nan.

color et interior ont une influence sur la valeur de revente, mais comme elle n’est pas indispensable à l’analyse globale de ce jeu de données, et que son imputation serait spéculative, j’ai choisi de conserver les valeurs manquantes. En effet, comme certaines couleurs sont effectivement plus recherchées que d’autres, il faut éviter d’imputer une couleur au hasard ou sur une intuition non vérifiable.

condition et odometer sont plus importants pour comprendre le prix, donc il vaut mieux les conserver via imputation.

### 2.5 Imputation de condition

d'abord :  le nombre de valeurs uniques, la répartition, et les mesures de dispersion.

In [298]:
condition = data['condition']

resume_condition = pd.DataFrame({
    'n_valeurs_uniques': [condition.nunique()],
    'nb_non_null': [condition.notna().sum()],
    'moyenne': [condition.mean()],
    'mediane': [condition.median()],
    'ecart_type': [condition.std()],
    'variance': [condition.var()],
    'min': [condition.min()],
    'q1': [condition.quantile(0.25)],
    'q3': [condition.quantile(0.75)],
    'max': [condition.max()]
})

resume_condition

,n_valeurs_uniques,nb_non_null,moyenne,mediane,ecart_type,variance,min,q1,q3,max
0,41,547005,30.67,35.00,13.40,179.64,1.00,23.00,42.00,49.00


* 41 valeurs uniques : condition n’est pas une simple échelle de 1 à 5 ; elle prend beaucoup plus de niveaux, donc c’est une variable quasi continue ou au moins très fine
* Médiane 35 > moyenne 30.67 : la distribution est un peu tirée vers les valeurs plus faibles, ce qui suggère qu’il y a plusieurs véhicules dans des états moyens ou dégradés.
* Écart-type 13.41 : la dispersion est assez forte, donc les véhicules sont répartis sur une large gamme d’états.


* la moyenne est trop sensible pour être prise comme valeur à la place de Nan.
* Plus judicieux de prendre la médiane, mais par groupe de voiture (on n'attend pas la même chose d'une fiat panda d'un SUV)

D'abord, je note les valeurs que je vais modifier à l'aide d'un drapeau

In [299]:
data['condition_was_nan'] = data['condition'].isna()

Je regarde si le groupe de voiture former par l'année, la marque et le model est pertinant

In [300]:
group_cols = ['year','make','model']
grp = data.groupby(group_cols)['condition']

# effectifs non-nuls par groupe
grp_count = grp.count().rename('count_non_null')
# médiane et IQR par groupe
grp_median = grp.median().rename('median')
grp_q1 = grp.quantile(0.25).rename('q1')
grp_q3 = grp.quantile(0.75).rename('q3')
grp_iqr = (grp_q3 - grp_q1).rename('iqr')

grp_stats = pd.concat([grp_count, grp_median, grp_q1, grp_q3, grp_iqr], axis=1).reset_index()
grp_stats

,year,make,model,count_non_null,median,q1,q3,iqr
0,1984,chevrolet,corvette,1,2.00,2.00,2.00,0.00
1,1985,chevrolet,corvette,1,4.00,4.00,4.00,0.00
2,1986,chevrolet,corvette,1,4.00,4.00,4.00,0.00
3,1986,mercedes,420sel,0,NaN,NaN,NaN,NaN
4,1987,mercedes,300e,1,2.00,2.00,2.00,0.00
...,...,...,...,...,...,...,...,...
5372,2015,volvo,s60,210,42.00,39.00,44.00,5.00
5373,2015,volvo,s80,1,42.00,42.00,42.00,0.00
5374,2015,volvo,v60,83,42.00,38.00,44.00,6.00
5375,2015,volvo,xc60,78,45.00,43.00,48.00,5.00


distribution des tailles de groupe

In [301]:
counts = grp_stats['count_non_null']
counts
print(counts.describe(percentiles=[0.25,0.5,0.75,0.9,0.99]))

count   5377.00
mean      99.81
std      294.77
min        0.00
25%        6.00
50%       25.00
75%       84.00
90%      213.40
99%     1274.64
max     8345.00
Name: count_non_null, dtype: float64


La distribution des tailles de groupes année, marque, model est très asymétrique : la médiane est de 25  observations, tandis que la moyenne est proche de 99 en raison de quelques groupes très volumineux. Un seuil de 10 observations par groupe est donc conservateur mais raisonnable, car il reste inférieur à la médiane et permet de conserver un grand nombre de groupes exploitables pour l’imputation.

Je vais donc tester avec un seuil de 10

In [302]:
seuil = 10

groupes_ok = grp_stats[grp_stats['count_non_null'] >= seuil]

print('Nombre de groupes avec au moins', seuil, 'ligne :', len(groupes_ok))
print('Nombre de lignes couvertes :', groupes_ok['count_non_null'].sum())
print('Part des lignes couvertes :', groupes_ok['count_non_null'].sum() / len(data) * 100, '%')

Nombre de groupes avec au moins 10 ligne : 3602
Nombre de lignes couvertes : 530165
Part des lignes couvertes : 94.87579612705105 %


Ces groupes sont-ils pertinants ?

Si IQR médian des groupes retenus est beaucoup plus petit que IQR global odometer, alors le groupe année, marque, modele est bon.

Si les deux sont proches, le groupe n’explique pas bien la valeur de l'attribut condition

In [303]:
# dispersion globale de condition
global_q1 = data['condition'].quantile(0.25)
global_q3 = data['condition'].quantile(0.75)
global_iqr = global_q3 - global_q1

# dispersion global
print("IQR global condition :", global_iqr)

# on regarde seulement les groupes qui ont au moins 10 valeurs non nulles
grp_10 = grp_stats[grp_stats['count_non_null'] >= 10]

print("Nombre de groupes retenus :", len(grp_10))
print("IQR médian des groupes retenus :", grp_10['iqr'].median())
print("IQR moyen des groupes retenus :", grp_10['iqr'].mean())

# résumé de la dispersion des groupes retenus
print(grp_10['iqr'].describe())

IQR global condition : 19.0
Nombre de groupes retenus : 3602
IQR médian des groupes retenus : 11.5
IQR moyen des groupes retenus : 12.559689061632426
count   3602.00
mean      12.56
std        6.71
min        0.00
25%        9.00
50%       11.50
75%       14.00
max       44.00
Name: iqr, dtype: float64




IQR groupe << IQR global : imputation par groupe pertinente.
 J'impute donc avec ce groupe

In [304]:
med_group = data.groupby(['make', 'model', 'year'])['condition'].transform('median')
data['condition'] = data['condition'].fillna(med_group)

In [305]:
data[data['condition'].isna()]  #verification du résultat


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate,condition_was_nan
2082,2011,NaN,NaN,NaN,NaN,automatic,1ftlr1fe5bpa06821,ca,NaN,1.00,white,—,onemain rem/e motorcars,18300.00,11500.00,Thu Dec 18 2014 12:00:00 GMT-0800 (PST),True
3420,2007,NaN,NaN,NaN,NaN,manual,3gnda23d37s503802,ca,NaN,76597.00,white,gray,toyota financial services,5475.00,1500.00,Wed Dec 17 2014 13:30:00 GMT-0800 (PST),True
4445,2004,NaN,NaN,NaN,NaN,manual,jm1fe173940111590,ca,NaN,101072.00,black,gray,mazda of escondido,3475.00,1100.00,Wed Dec 17 2014 13:30:00 GMT-0800 (PST),True
5159,1996,toyota,land cruiser,base,suv,NaN,jt3hj85j6t0123780,ca,NaN,177547.00,red,—,symbolic motor cars,3525.00,3500.00,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),True
5227,1997,ford,f150,4x4 ext lariat,NaN,NaN,1ftex18l9vkc18584,ca,NaN,115395.00,burgundy,—,walters auto sales/service inc,2125.00,2400.00,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41314,1995,ford,windstar,fwd,NaN,automatic,2fmda5147sba89298,tx,NaN,128219.00,purple,gray,big johns motor co,600.00,900.00,Thu Dec 18 2014 11:00:00 GMT-0800 (PST),True
41336,1985,NaN,NaN,NaN,NaN,automatic,1g6el5787fe612281,il,NaN,67786.00,brown,—,phillips chevrolet of lansing inc,2975.00,550.00,Thu Dec 18 2014 10:00:00 GMT-0800 (PST),True
41337,1990,chevrolet,1500,4x2 ext base,NaN,NaN,2gcec19k2l1242286,mo,NaN,156215.00,white,—,animal protective association of missouri,550.00,800.00,Mon Dec 22 2014 11:00:00 GMT-0800 (PST),True
41340,1987,NaN,NaN,NaN,NaN,automatic,1g1yy2187h5126181,il,NaN,107955.00,orange,—,lexus of merrillville,3950.00,3000.00,Thu Dec 18 2014 10:00:00 GMT-0800 (PST),True


Les médianes ou make ou model ou year sont à NaN vont avoir une médiane à Nan, regardons ce qui se passe quand ce n'est pas le cas. Le groupe serait-il former que de voiture, dont on ne connaît pas la condition ?

In [306]:
#les valeurs NaN pour condition après traitement et avec le groupe de complètelment defini
# note : pas de year inconnu
condition_na = data[data['condition'].isna() & data['make'].notna() & data['model'].notna()][['condition', 'year', 'make', 'model']]
condition_na

,condition,year,make,model
5159,NaN,1996,toyota,land cruiser
5227,NaN,1997,ford,f150
5228,NaN,1996,ford,windstar
5271,NaN,1989,toyota,pickup
7724,NaN,1997,pontiac,grand
...,...,...,...,...
39218,NaN,2009,gmc,sr
39602,NaN,2008,pontiac,montana
39936,NaN,2007,suzuki,swift
41314,NaN,1995,ford,windstar


In [307]:
#est-ce qu'il existe dans data des valeurs qui ont une condition pour ces groupes
result = data[data['condition'].notna()].merge(condition_na, on=['year', 'make', 'model'], how='inner')
print(len(result))
len(data)

0


558799

Je conserve uniquement les lignes assez complètes pour être rattachées à un groupe make/model/year fiable (et dans les NaN restant je met la médiane total). Quand make ou model est manquant, le rattachement devient ambigu, donc ces lignes sont exclues de l’imputation par groupe.

In [308]:
#suppression
data = data[~(data['condition'].isna() & (data['make'].isna() | data['model'].isna()))].copy()
#imputation
data['condition'] = data['condition'].fillna(data['condition'].median())
#verification
data['condition'].isna().sum()

np.int64(0)

Nous avons bien des conditions NaN après traitement qui ont soit make soit model à NaN ou qui forme un groupe de voiture dont la condition est inconnue elle sont 196 dont 109 pour les groupe de voiture dont la condition est inconnue


### 2.6 Imputation d'odometer

d'abord : le nombre de valeurs uniques, la répartition, et les mesures de dispersion.

In [309]:
odometer = data['odometer']

resume_odometer = pd.DataFrame({
    'n_valeurs_uniques': [odometer.nunique()],
    'nb_non_null': [odometer.notna().sum()],
    'moyenne': [odometer.mean()],
    'mediane': [odometer.median()],
    'ecart_type': [odometer.std()],
    'variance': [odometer.var()],
    'min': [odometer.min()],
    'q1': [odometer.quantile(0.25)],
    'q3': [odometer.quantile(0.75)],
    'max': [odometer.max()]
})

resume_odometer

,n_valeurs_uniques,nb_non_null,moyenne,mediane,ecart_type,variance,min,q1,q3,max
0,172252,558619,68311.62,52250.00,53387.16,2850188607.27,1.00,28370.00,99098.00,999999.00


In [310]:
print('Nombre de lignes : ', len(data),' odometer Nana', data['odometer'].isna().sum()/len(data))

Nombre de lignes :  558712  odometer Nana 0.00016645427340024915


* 172193 valeurs uniques : odometer prend beaucoup de valeur, donc c’est une variable quasi continue ou au moins très fine
* Médiane 52217 < moyenne 68301.3 : la distribution est un tirée vers les valeurs plus élevé, ce qui suggère qu’il y a plusieurs véhicules avec un fort kilometrage.
* Écart-type 53398.39 : la dispersion est forte, donc les véhicules sont répartis sur de nombreux kilométrage.


* la moyenne est trop sensible pour être prise comme valeur à la place de NaN.
* Plus judicieux de prendre la médiane, mais par groupe de voiture

* je regarde à quoi correspond le max.

In [311]:
data[data['odometer'] == 999999.0 ]

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate,condition_was_nan
275,2013,hyundai,elantra coupe,gs,elantra coupe,automatic,kmhdh6ae8du017422,ca,1.00,999999.00,blue,gray,hyundai motor finance,8025.00,2500.00,Tue Jan 27 2015 04:00:00 GMT-0800 (PST),False
4626,2003,chevrolet,silverado 1500,ls,extended cab,automatic,1gcec19v43e225059,ca,2.00,999999.00,gray,gray,800 loan mart,1425.00,700.00,Tue Dec 16 2014 13:00:00 GMT-0800 (PST),False
13317,2009,chevrolet,cobalt,lt,coupe,automatic,1g1at18h797165360,tx,27.00,999999.00,white,gray,mei finance,3375.00,400.00,Thu Dec 18 2014 14:00:00 GMT-0800 (PST),True
13480,2009,dodge,charger,base,sedan,automatic,2b3ka43d49h578284,il,1.00,999999.00,black,gray,santander consumer,3850.00,1700.00,Tue Dec 23 2014 13:00:00 GMT-0800 (PST),False
13568,2009,dodge,charger,base,sedan,automatic,2b3ka43dx9h521300,tx,1.00,999999.00,blue,black,santander consumer,4150.00,5500.00,Thu Jan 08 2015 14:10:00 GMT-0800 (PST),False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
489833,2008,chrysler,300,touring,sedan,automatic,2c3la53g88h288734,ca,27.00,999999.00,silver,beige,chrysler capital,4325.00,1200.00,Tue Jun 02 2015 06:00:00 GMT-0700 (PDT),False
490461,2002,ford,expedition,eddie bauer,suv,automatic,1fmru17l72la49923,tx,2.00,999999.00,black,gray,titlemax/leon valley tx1,600.00,600.00,Wed Jun 03 2015 03:20:00 GMT-0700 (PDT),False
501478,2011,volkswagen,jetta,sel,sedan,NaN,3vwlx7aj1bm387406,tx,1.00,999999.00,white,tan,santander consumer,8600.00,1100.00,Wed Jun 03 2015 05:30:00 GMT-0700 (PDT),False
513219,2006,ford,taurus,sel,sedan,automatic,1fafp56u06a197709,oh,19.00,999999.00,gray,gray,car now acceptance co/columbus,175.00,400.00,Thu Jun 11 2015 02:00:00 GMT-0700 (PDT),False


Cela semble être des outliers statistique, mais rien ne semble les caractériser. Je ne vais donc pas y toucher, ils ne semblent pas faux




Je note les valeurs que je vais modifier à l'aide d'un drapeau

In [312]:
data['odometer_was_nan'] = data['odometer'].isna()

Je regarde si le groupe de voiture former par l'année, la marque et le model est pertinant en excluant les odometers qui semblent être des outliers.

In [313]:
group_cols = ['year','make','model']
grp = data.groupby(group_cols)['odometer']

# effectifs non-nuls par groupe
grp_count = grp.count().rename('count_non_null')
# médiane et IQR par groupe
grp_median = grp.median().rename('median')
grp_q1 = grp.quantile(0.25).rename('q1')
grp_q3 = grp.quantile(0.75).rename('q3')
grp_iqr = (grp_q3 - grp_q1).rename('iqr')

grp_stats = pd.concat([grp_count, grp_median, grp_q1, grp_q3, grp_iqr], axis=1).reset_index()
grp_stats


,year,make,model,count_non_null,median,q1,q3,iqr
0,1984,chevrolet,corvette,1,46891.00,46891.00,46891.00,0.00
1,1985,chevrolet,corvette,2,83193.50,76807.75,89579.25,12771.50
2,1986,chevrolet,corvette,1,12466.00,12466.00,12466.00,0.00
3,1986,mercedes,420sel,1,72250.00,72250.00,72250.00,0.00
4,1987,mercedes,300e,1,151242.00,151242.00,151242.00,0.00
...,...,...,...,...,...,...,...,...
5372,2015,volvo,s60,210,13542.00,11103.00,16177.75,5074.75
5373,2015,volvo,s80,1,12497.00,12497.00,12497.00,0.00
5374,2015,volvo,v60,83,12349.00,10741.50,15433.00,4691.50
5375,2015,volvo,xc60,78,13861.00,11071.25,16890.25,5819.00


distribution des tailles de groupe

In [314]:
counts = grp_stats['count_non_null']
counts
print(counts.describe(percentiles=[0.25,0.5,0.75,0.9,0.99]))

count   5377.00
mean     101.97
std      296.01
min        1.00
25%        6.00
50%       26.00
75%       86.00
90%      220.40
99%     1278.16
max     8353.00
Name: count_non_null, dtype: float64


La distribution des tailles de groupes année, marque, model est très asymétrique : la médiane est de 26 observations, tandis que la moyenne est proche de 102 en raison de quelques groupes très volumineux. Un seuil de 10 observations par groupe est donc conservateur mais raisonnable, car il reste inférieur à la médiane et permet de conserver un grand nombre de groupes exploitables pour l’imputation.

Je vais donc tester avec un seuil de 10

In [315]:
seuil = 10

groupes_ok = grp_stats[grp_stats['count_non_null'] >= seuil]

print('Nombre de groupes avec au moins', seuil, 'ligne :', len(groupes_ok))
print('Nombre de lignes couvertes :', groupes_ok['count_non_null'].sum())
print('Part des lignes couvertes :', groupes_ok['count_non_null'].sum() / len(data) * 100, '%')

Nombre de groupes avec au moins 10 ligne : 3657
Nombre de lignes couvertes : 541757
Part des lignes couvertes : 96.9653417145148 %


Ces groupes sont-ils pertinants ?

Si IQR médian des groupes retenus est beaucoup plus petit que IQR global odometer, alors le groupe année, marque, modele est bon.

Si les deux sont proches, le groupe n’explique pas bien l'odometer


In [316]:
# dispersion globale de odometer
global_q1 = data['odometer'].quantile(0.25)
global_q3 = data['odometer'].quantile(0.75)
global_iqr = global_q3 - global_q1

# dispersion global
print("IQR global odometer :", global_iqr)

# on regarde seulement les groupes qui ont au moins 10 valeurs non nulles
grp_10 = grp_stats[grp_stats['count_non_null'] >= 10]

print("Nombre de groupes retenus :", len(grp_10))
print("IQR médian des groupes retenus :", grp_10['iqr'].median())
print("IQR moyen des groupes retenus :", grp_10['iqr'].mean())

# résumé de la dispersion des groupes retenus
print(grp_10['iqr'].describe())

IQR global odometer : 70728.0
Nombre de groupes retenus : 3657
IQR médian des groupes retenus : 37259.0
IQR moyen des groupes retenus : 38262.15545529122
count     3657.00
mean     38262.16
std      20243.43
min       1494.50
25%      23463.50
50%      37259.00
75%      49740.00
max     168559.50
Name: iqr, dtype: float64


IQR groupe << IQR global : imputation par groupe pertinente.

In [317]:
tmp = data[['make', 'model', 'year', 'odometer']].copy()
med_group = tmp.groupby(['make', 'model', 'year'])['odometer'].transform('median')

data['odometer'] = data['odometer'].fillna(med_group)

In [318]:
data[data['odometer'].isna()]  #verification du résultat

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate,condition_was_nan,odometer_was_nan
2690,2010,NaN,NaN,NaN,NaN,automatic,2fabp7bv5ax113795,ca,2.00,NaN,white,black,enterprise fm exchange/tra/lease,3700.00,2200.00,Tue Jan 27 2015 04:00:00 GMT-0800 (PST),False,True
392853,2000,NaN,NaN,NaN,NaN,automatic,1j4fa49s7yp760863,pa,19.00,NaN,green,gray,foulke management corp,6700.00,4850.00,Tue Mar 03 2015 01:30:00 GMT-0800 (PST),False,True
490195,2006,NaN,NaN,NaN,NaN,manual,ja3aj16e16u070827,pr,21.00,NaN,gray,gray,oriental bank,2900.00,1700.00,Thu Jun 04 2015 04:00:00 GMT-0700 (PDT),False,True


Il reste 3 voitures dont on ne connaît pas le resultat, toutes n'ont ni marque, ni model. Je conserve uniquement les lignes assez complètes pour être rattachées à un groupe make/model/year, je les supprimes

In [319]:
data = data.dropna(subset='odometer')

In [320]:
data[data['odometer'].isna()]  #verification du résultat

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate,condition_was_nan,odometer_was_nan


### 2.7 Gestion des dates

Est-ce que les dates sont toutes au même format ?

In [321]:
pattern = r"^[A-Z][a-z]{2} [A-Z][a-z]{2} \d{2} \d{4} \d{2}:\d{2}:\d{2} GMT[-+]\d{4} \([A-Z]{2,5}\)$"

mask_ok = data['saledate'].astype(str).str.match(pattern)
data[~mask_ok]


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate,condition_was_nan,odometer_was_nan


Oui.
On ne va pas extraire le fuseau horaire. Ce qui nous intéresse c'est la date telle que la vie l'acheteur

Je dispatche la date pour mon analyse

In [322]:
# Extraire juste la partie principale (jour, mois, an, heure)
data_clean = data['saledate'].str.extract(r'(\w+ \w+ \d+ \d{4} \d+:\d+:\d+)')[0]

data['saledate_dt'] = pd.to_datetime(data_clean, format='%a %b %d %Y %H:%M:%S')

data['day'] = data['saledate_dt'].dt.strftime('%a')
data['month'] = data['saledate_dt'].dt.strftime('%b')
data['year_s'] = data['saledate_dt'].dt.year

data = data.drop(columns=['saledate_dt'])
data.head()


/tmp/ipykernel_5434/4087898822.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_5434/4087898822.py:6: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_5434/4087898822.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_5434

,year,make,model,trim,body,transmission,vin,state,condition,odometer,...,interior,seller,mmr,sellingprice,saledate,condition_was_nan,odometer_was_nan,day,month,year_s
0,2015,kia,sorento,lx,suv,automatic,5xyktca69fg566472,ca,5.00,16639.00,...,black,kia motors america inc,20500.00,21500.00,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),False,False,Tue,Dec,2014
1,2015,kia,sorento,lx,suv,automatic,5xyktca69fg561319,ca,5.00,9393.00,...,beige,kia motors america inc,20800.00,21500.00,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),False,False,Tue,Dec,2014
2,2014,bmw,3 series,328i sulev,sedan,automatic,wba3c1c51ek116351,ca,45.00,1331.00,...,black,financial services remarketing (lease),31900.00,30000.00,Thu Jan 15 2015 04:30:00 GMT-0800 (PST),False,False,Thu,Jan,2015
3,2015,volvo,s60,t5,sedan,automatic,yv1612tb4f1310987,ca,41.00,14282.00,...,black,volvo na rep/world omni,27500.00,27750.00,Thu Jan 29 2015 04:30:00 GMT-0800 (PST),False,False,Thu,Jan,2015
4,2014,bmw,6 series gran coupe,650i,sedan,automatic,wba6b2c57ed129731,ca,43.00,2641.00,...,black,financial services remarketing (lease),66000.00,67000.00,Thu Dec 18 2014 12:30:00 GMT-0800 (PST),False,False,Thu,Dec,2014


In [323]:
data = data.reset_index(drop=False)  # crée une colonne 'index'

## 3. Axe d'analyse du marché

### 3.1 Prix et valeur du marché

* Calcul de la marge

marge = sellingprice - mmr

marge_pct = (marge /mnr)*100

In [324]:
data['marge']= data['sellingprice']-data['mmr']
data['marge_pct'] = data['marge']/data['mmr']*100
data

,index,year,make,model,trim,body,transmission,vin,state,condition,...,mmr,sellingprice,saledate,condition_was_nan,odometer_was_nan,day,month,year_s,marge,marge_pct
0,0,2015,kia,sorento,lx,suv,automatic,5xyktca69fg566472,ca,5.00,...,20500.00,21500.00,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),False,False,Tue,Dec,2014,1000.00,4.88
1,1,2015,kia,sorento,lx,suv,automatic,5xyktca69fg561319,ca,5.00,...,20800.00,21500.00,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),False,False,Tue,Dec,2014,700.00,3.37
2,2,2014,bmw,3 series,328i sulev,sedan,automatic,wba3c1c51ek116351,ca,45.00,...,31900.00,30000.00,Thu Jan 15 2015 04:30:00 GMT-0800 (PST),False,False,Thu,Jan,2015,-1900.00,-5.96
3,3,2015,volvo,s60,t5,sedan,automatic,yv1612tb4f1310987,ca,41.00,...,27500.00,27750.00,Thu Jan 29 2015 04:30:00 GMT-0800 (PST),False,False,Thu,Jan,2015,250.00,0.91
4,4,2014,bmw,6 series gran coupe,650i,sedan,automatic,wba6b2c57ed129731,ca,43.00,...,66000.00,67000.00,Thu Dec 18 2014 12:30:00 GMT-0800 (PST),False,False,Thu,Dec,2014,1000.00,1.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
558704,558832,2015,kia,k900,luxury,sedan,NaN,knalw4d4xf6019304,in,45.00,...,35300.00,33000.00,Thu Jul 09 2015 07:00:00 GMT-0700 (PDT),False,False,Thu,Jul,2015,-2300.00,-6.52
558705,558833,2012,ram,2500,power wagon,crew cab,automatic,3c6td5et6cg112407,wa,5.00,...,30200.00,30800.00,Wed Jul 08 2015 09:30:00 GMT-0700 (PDT),False,False,Wed,Jul,2015,600.00,1.99
558706,558834,2012,bmw,x5,xdrive35d,suv,automatic,5uxzw0c58cl668465,ca,48.00,...,29800.00,34000.00,Wed Jul 08 2015 09:30:00 GMT-0700 (PDT),False,False,Wed,Jul,2015,4200.00,14.09
558707,558835,2015,nissan,altima,2.5 s,sedan,automatic,1n4al3ap0fc216050,ga,38.00,...,15100.00,11100.00,Thu Jul 09 2015 06:45:00 GMT-0700 (PDT),False,False,Thu,Jul,2015,-4000.00,-26.49


* distribution des marges

In [325]:
print(data['marge'].describe())

count   558709.00
mean      -157.94
std       1759.00
min     -87750.00
25%       -800.00
50%        -50.00
75%        650.00
max     207200.00
Name: marge, dtype: float64


In [326]:
print(data['marge_pct'].describe(percentiles=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]))

count   558709.00
mean        -0.66
std         38.86
min       -100.00
10%        -21.79
20%        -10.42
30%         -5.42
40%         -2.50
50%         -0.42
60%          1.55
70%          4.00
80%          7.61
90%         15.38
max       8033.33
Name: marge_pct, dtype: float64


Je vais m'interresser au marge par pourcentage de la côte afin de pouvoir comparer les marges entre elles
Il y a peu de marge de faite, au moins la moitié des ventes se fait avec une marge négative.
Les outliers les plus petits représentent 20% des ventes (en dessous ou égale à -11%) et les outliers les plus grands représentent 10% des ventes (plus de 15%)
On va s'intéresser au vente sous la côte

In [327]:
print('Il y a ', data[data['marge']<0]['marge'].count(), 'ventes qui se font sous la côte')
print('Soit', data[data['marge']<0]['marge'].count()/len(data)*100, '% des ventes')

Il y a  286352 ventes qui se font sous la côte
Soit 51.25244089499185 % des ventes


* segmentation des marges et du volume de voiture vendue

** par marque

In [328]:
data = data.reset_index()
marge_make=data.groupby('make',as_index=False).agg(marge_mediane_pct=('marge_pct','median'), marge_mediane=('marge','median'), volume=('index', 'count'))
marge_make.sort_values('marge_mediane_pct', ascending=False, inplace=True)


In [329]:
# le bottom 10 en fonction du pourcentage
marge_make.tail(10)

,make,marge_mediane_pct,marge_mediane,volume
50,plymouth,-4.00,-125.00,27
45,mercury,-5.17,-150.00,2023
51,pontiac,-5.54,-175.00,4524
29,isuzu,-7.69,-100.00,204
13,dodge tk,-12.00,-75.00,1
49,oldsmobile,-14.29,-125.00,384
19,ford tk,-19.42,-675.00,1
21,geo,-20.00,-100.00,19
11,daewoo,-40.00,-200.00,3
14,dot,-41.18,-350.00,1


In [330]:
# le top 10 en fonction du pourcentage
marge_make.head(10)

,make,marge_mediane_pct,marge_mediane,volume
1,airstream,140.68,41500.00,1
41,mazda tk,20.93,1125.00,1
27,hyundai tk,12.00,225.00,1
8,chev truck,3.90,75.00,1
25,hummer,2.26,300.00,805
23,gmc truck,1.96,175.00,11
2,aston martin,1.74,800.00,25
38,lotus,1.24,500.00,1
60,suzuki,0.90,25.00,1078
0,acura,0.25,25.00,5926


In [331]:
marge_make.sort_values('volume', ascending=False, inplace=True)

In [332]:
# le bottom 10 en fonction du volume
marge_make.tail(10)

,make,marge_mediane_pct,marge_mediane,volume
11,daewoo,-40.00,-200.00,3
43,mercedes-b,-3.75,-450.00,2
41,mazda tk,20.93,1125.00,1
1,airstream,140.68,41500.00,1
27,hyundai tk,12.00,225.00,1
38,lotus,1.24,500.00,1
8,chev truck,3.90,75.00,1
13,dodge tk,-12.00,-75.00,1
19,ford tk,-19.42,-675.00,1
14,dot,-41.18,-350.00,1


In [333]:
# le top 10 en fonction du volume
marge_make.head(10)

,make,marge_mediane_pct,marge_mediane,volume
18,ford,-0.62,-100.00,93996
9,chevrolet,-0.37,-50.00,60587
48,nissan,-0.40,-50.00,54017
62,toyota,-0.40,-50.00,39966
12,dodge,-0.53,-50.00,30953
24,honda,0.00,0.00,27351
26,hyundai,0.00,0.00,21831
5,bmw,-0.37,-75.00,20793
32,kia,0.00,0.00,18082
10,chrysler,-0.78,-100.00,17483


** par carrosserie

In [334]:
marge_body=data.groupby('body',as_index=False).agg(marge_mediane_pct=('marge_pct','median'), marge_mediane=('marge','median'), volume=('index', 'count'))
marge_body.sort_values('marge_mediane_pct', ascending=False, inplace=True)

In [335]:
# le bottom 10 en fonction du pourcentage
marge_body.tail(10)

,body,marge_mediane_pct,marge_mediane,volume
27,mega cab,-0.78,-300.00,111
34,regular cab,-0.83,-100.00,4850
32,quad cab,-1.11,-175.00,4095
40,transit van,-1.18,-300.00,19
21,g37 coupe,-2.16,-350.00,12
44,xtracab,-2.20,-137.50,44
11,cts-v coupe,-2.60,-1000.00,35
23,granturismo convertible,-3.85,-3000.00,13
33,ram van,-7.69,-100.00,1
2,cab plus,-21.25,-500.00,4


In [336]:
# le top 10 en fonction du pourcentage
marge_body.head(10)

,body,marge_mediane_pct,marge_mediane,volume
12,cts-v wagon,19.39,8200.00,1
3,cab plus 4,10.81,787.50,6
10,cts wagon,5.18,1100.00,14
41,tsx sport wagon,3.98,700.00,36
4,club cab,3.30,125.00,178
30,q60 convertible,2.16,750.00,42
22,genesis coupe,2.08,300.00,294
29,promaster cargo van,1.32,300.00,59
35,regular-cab,1.18,200.00,15
0,access cab,0.68,100.00,294


In [337]:
marge_body.sort_values('volume', ascending=False, inplace=True)

In [338]:
# le bottom 10 en fonction du volume
marge_body.tail(10)

,body,marge_mediane_pct,marge_mediane,volume
20,g37 convertible,-0.26,-50.00,20
40,transit van,-1.18,-300.00,19
35,regular-cab,1.18,200.00,15
10,cts wagon,5.18,1100.00,14
23,granturismo convertible,-3.85,-3000.00,13
21,g37 coupe,-2.16,-350.00,12
3,cab plus 4,10.81,787.50,6
2,cab plus,-21.25,-500.00,4
12,cts-v wagon,19.39,8200.00,1
33,ram van,-7.69,-100.00,1


In [339]:
# le top 10 en fonction du volume
marge_body.head(10)

,body,marge_mediane_pct,marge_mediane,volume
36,sedan,-0.51,-50.00,241332
39,suv,-0.31,-50.00,143844
24,hatchback,-0.43,-50.00,26237
28,minivan,-0.51,-50.00,25529
6,coupe,-0.29,-50.00,17752
7,crew cab,0.00,0.00,16394
43,wagon,-0.39,-50.00,16128
5,convertible,-0.25,-25.00,10476
38,supercrew,-0.41,-100.00,9033
19,g sedan,-0.26,-50.00,7417


** En fonction de l'état

L'état varie de 0 à 49, je vais donc faire 5 catégorie (0 de 0 à 9, 1 de 10 à 19, etc.)

In [340]:
def categorie_condition(nombre):
  if nombre < 10 :
    return 0
  elif nombre < 20 :
    return 1
  elif nombre < 30 :
    return 2
  elif nombre < 40 :
    return 3
  else :
    return 4

condition_categorie = data['condition'].apply(categorie_condition)
data.loc[:,'condition_categorie']= condition_categorie
data.head()


,level_0,index,year,make,model,trim,body,transmission,vin,state,...,sellingprice,saledate,condition_was_nan,odometer_was_nan,day,month,year_s,marge,marge_pct,condition_categorie
0,0,0,2015,kia,sorento,lx,suv,automatic,5xyktca69fg566472,ca,...,21500.00,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),False,False,Tue,Dec,2014,1000.00,4.88,0
1,1,1,2015,kia,sorento,lx,suv,automatic,5xyktca69fg561319,ca,...,21500.00,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),False,False,Tue,Dec,2014,700.00,3.37,0
2,2,2,2014,bmw,3 series,328i sulev,sedan,automatic,wba3c1c51ek116351,ca,...,30000.00,Thu Jan 15 2015 04:30:00 GMT-0800 (PST),False,False,Thu,Jan,2015,-1900.00,-5.96,4
3,3,3,2015,volvo,s60,t5,sedan,automatic,yv1612tb4f1310987,ca,...,27750.00,Thu Jan 29 2015 04:30:00 GMT-0800 (PST),False,False,Thu,Jan,2015,250.00,0.91,4
4,4,4,2014,bmw,6 series gran coupe,650i,sedan,automatic,wba6b2c57ed129731,ca,...,67000.00,Thu Dec 18 2014 12:30:00 GMT-0800 (PST),False,False,Thu,Dec,2014,1000.00,1.52,4


In [341]:
marge_condition=data.groupby('condition_categorie',as_index=False).agg(marge_mediane_pct=('marge_pct','median'), marge_mediane=('marge','median'),volume=('index','count'))
marge_condition.sort_values('marge_mediane_pct', ascending=False, inplace=True)
marge_condition


,condition_categorie,marge_mediane_pct,marge_mediane,volume
4,4,1.84,300.00,160338
3,3,0.26,25.00,164336
0,0,-2.13,-250.00,70415
2,2,-4.68,-375.00,118006
1,1,-16.55,-700.00,45614


Les marges semblent suivent les catégories de condition. Plus la voiture est en bonne condition, plus on a de chance d'avoir une marge positive.

 On va tester la corrélation en la condition (et non sa catégorie) et le pourcentage de marge par rapport au prix mmr

In [342]:
pd.DataFrame(pearsonr(data['marge_pct'], data['condition']), index=['pearson_coeff','p-value'], columns=['resultat_test'])

,resultat_test
pearson_coeff,0.11
p-value,0.00


La p-value est inférieur à 5% on rejette donc le fait que la marge en pourcentage et la contion soit indépendante

In [343]:
pd.DataFrame(pearsonr(data['marge'], data['condition']), index=['pearson_coeff','p-value'], columns=['resultat_test'])

,resultat_test
pearson_coeff,0.23
p-value,0.00


La p-value est inférieur à 5% on rejette donc le fait que la marge en dolloar et la condition soit indépendante

In [344]:
condition_data = data[['marge', 'marge_pct', 'condition']].copy()

In [345]:
# Affichage de la heatmap de ces corrélations
condition_data.corr().style.background_gradient(cmap='Reds')

,marge,marge_pct,condition
marge,1.000000,0.375957,0.231066
marge_pct,0.375957,1.000000,0.114236
condition,0.231066,0.114236,1.000000


Néanmois les corrélations restent faible.

On observe sur un schéma :

In [346]:
data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558709 entries, 0 to 558708
Data columns (total 26 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   level_0              558709 non-null  int64  
 1   index                558709 non-null  int64  
 2   year                 558709 non-null  int64  
 3   make                 548498 non-null  object 
 4   model                548400 non-null  object 
 5   trim                 548148 non-null  object 
 6   body                 545604 non-null  object 
 7   transmission         493376 non-null  object 
 8   vin                  558709 non-null  object 
 9   state                558709 non-null  object 
 10  condition            558709 non-null  float64
 11  odometer             558709 non-null  float64
 12  color                557962 non-null  object 
 13  interior             557962 non-null  object 
 14  seller               558709 non-null  object 
 15  mmr              

In [347]:
data_condition_cat_0 = data[data['condition_categorie']==0].copy()
data_condition_cat_1 = data[data['condition_categorie']==1].copy()
data_condition_cat_2 = data[data['condition_categorie']==2].copy()
data_condition_cat_3 = data[data['condition_categorie']==3].copy()
data_condition_cat_4 = data[data['condition_categorie']==4].copy()

conditions = {
    'Condition 0': data_condition_cat_0['marge_pct'],
    'Condition 1': data_condition_cat_1['marge_pct'],
    'Condition 2': data_condition_cat_2['marge_pct'],
    'Condition 3': data_condition_cat_3['marge_pct'],
    'Condition 4': data_condition_cat_4['marge_pct'],
}

res = pd.DataFrame({
    'condition': list(conditions.keys()),
    'pct_positive': [100 * (s > 0).mean() for s in conditions.values()],
    'median_marge': [s.median() for s in conditions.values()],
    'mean_marge': [s.mean() for s in conditions.values()]})

fig = go.Figure()

# Barres
fig.add_trace(go.Bar(
    x=res['condition'],
    y=res['pct_positive'],
    marker_color='steelblue',
    text=res['pct_positive'].round(1).astype(str) + '%',
    textposition='outside'
))

# Ligne rouge à 50%
fig.add_hline(
    y=50,
    line_dash="dash",
    line_color="red",
    line_width=1,
    annotation_text="50%",
    annotation_position="right"
)

fig.update_layout(
    title="Part de marges positives par condition",
    yaxis=dict(title="% de marges positives", range=[0, 110]),
    xaxis=dict(title="Condition", tickangle=-30),
    showlegend=False
)

fig.show()

On constate que seul les voitures en très bon état (condition 4) permet de se faire dans plus de 50% des cas une marge positive par rapport à l'argus. On ne revend donc pas prioritairement pour se faire de l'argent.



Les volume semblent suivent les catégories de condition. Plus la voiture est en bonne condition, plus on a de chance de vendre beaucoup de voiture.


On va visualisé sur un schéma

In [348]:


fig = px.histogram(
    data,
    x="condition_categorie",
    title="Volume de ventes par condition",
    labels={
        "condition_categorie": "Condition",
        "count": "Nombre de ventes"
    },
)
fig.update_layout(
    xaxis=dict(
        type="category",
        categoryorder="array",          # ← ordre défini par categoryarray

        categoryarray=sorted(marge_condition["condition_categorie"].unique()),
        tickmode="array",
        tickvals=sorted(marge_condition["condition_categorie"].unique()),
        ticktext=["Condition " + str(i)
                  for i in sorted(marge_condition["condition_categorie"].unique())]
    ), showlegend=False

)
fig.show()

Visualisons avec un shéma, si les catégories avec plus de ventes ont aussi des marges différentes.

In [349]:
fig = px.scatter(
    marge_condition,
    x="condition_categorie",
    y="marge_mediane_pct",
    size="volume",
    color="condition_categorie",
    title="Volume de ventes vs Marge par condition",
    labels={
        "condition_categorie": "Condition",
        "marge_mediane_pct": "Marge médiane (%)",
        "volume": "Volume de ventes"
    },
    size_max=60
)
fig.update_layout(
    xaxis=dict(
        type="category",
        categoryorder="array",          # ← ordre défini par categoryarray
        categoryarray=sorted(marge_condition["condition_categorie"].unique()),
        tickmode="array",
        tickvals=sorted(marge_condition["condition_categorie"].unique()),
        ticktext=["Condition " + str(i)
                  for i in sorted(marge_condition["condition_categorie"].unique())]
    ),
    showlegend=False
)
fig.show()

Les voitures en meilleure condition ont un meilleur volume de vente et une marge médiane plutôt positive. Les voitures en très mauvais état, on néanmoins une faible décote.

** En fonction du kilometrage

In [350]:
print(data['odometer'].describe(percentiles=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]))

count   558709.00
mean     68321.26
std      53390.58
min          1.00
10%      15673.00
20%      23989.00
30%      32358.00
40%      40268.20
50%      52257.00
60%      67117.00
70%      88612.00
80%     111042.00
90%     142757.20
max     999999.00
Name: odometer, dtype: float64


L'état varie de 0 à 999999 (voir au dessus), je vais donc faire 11 catégories de 10000 en 10000, puis augmenter le pas

In [351]:
bins = [
    0, 10000, 20000, 30000, 40000, 50000, 60000, 80000, 100000,
    125000, 150000, 200000, 300000, 500000, 700000, 900000, 1000000
]

labels = [bins[i+1] for i in range(len(bins)-1)]

data['odometer_categorie'] = pd.cut(data["odometer"], bins=bins, labels=labels, right=False)
data['odometer_categorie'].value_counts(dropna=False)
data.head()



,level_0,index,year,make,model,trim,body,transmission,vin,state,...,saledate,condition_was_nan,odometer_was_nan,day,month,year_s,marge,marge_pct,condition_categorie,odometer_categorie
0,0,0,2015,kia,sorento,lx,suv,automatic,5xyktca69fg566472,ca,...,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),False,False,Tue,Dec,2014,1000.00,4.88,0,20000
1,1,1,2015,kia,sorento,lx,suv,automatic,5xyktca69fg561319,ca,...,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),False,False,Tue,Dec,2014,700.00,3.37,0,10000
2,2,2,2014,bmw,3 series,328i sulev,sedan,automatic,wba3c1c51ek116351,ca,...,Thu Jan 15 2015 04:30:00 GMT-0800 (PST),False,False,Thu,Jan,2015,-1900.00,-5.96,4,10000
3,3,3,2015,volvo,s60,t5,sedan,automatic,yv1612tb4f1310987,ca,...,Thu Jan 29 2015 04:30:00 GMT-0800 (PST),False,False,Thu,Jan,2015,250.00,0.91,4,20000
4,4,4,2014,bmw,6 series gran coupe,650i,sedan,automatic,wba6b2c57ed129731,ca,...,Thu Dec 18 2014 12:30:00 GMT-0800 (PST),False,False,Thu,Dec,2014,1000.00,1.52,4,10000


In [352]:


marge_odometer=data.groupby('odometer_categorie',as_index=False,observed=True).agg(marge_mediane_pct=('marge_pct','median'), marge_mediane=('marge','median'), volume=('index', 'count'))
marge_odometer.sort_values('marge_mediane_pct')
marge_odometer

,odometer_categorie,marge_mediane_pct,marge_mediane,volume
0,10000,0.00,0.00,25757
1,20000,-0.32,-50.00,58422
2,30000,-0.27,-50.00,66790
3,40000,0.00,0.00,70817
4,50000,0.00,0.00,48326
5,60000,0.00,0.00,41018
6,80000,0.01,1.00,57895
7,100000,-0.31,-25.00,52405
8,125000,-3.25,-200.00,53785
9,150000,-5.62,-225.00,36661


Y-a-t-il une corrélation ? Entre la marge en pourcentage et l'odometer (pas la catégorie)

In [353]:
pd.DataFrame(pearsonr(data['marge_pct'], data['odometer']), index=['pearson_coeff','p-value'], columns=['resultat_test'])

,resultat_test
pearson_coeff,0.06
p-value,0.00


La p-value est inférieur à 5% on rejette donc le fait que la marge en pourcentage et l'odometer soit indépendante

In [354]:
odometer_data = data[['marge', 'marge_pct', 'odometer']].copy()

In [355]:
# Affichage de la heatmap de ces corrélations
odometer_data.corr().style.background_gradient(cmap='Reds')

,marge,marge_pct,odometer
marge,1.000000,0.375957,0.008240
marge_pct,0.375957,1.000000,0.056121
odometer,0.008240,0.056121,1.000000


Encore une fois les corrélations restent faible.

In [356]:
marge_odometer = marge_odometer.sort_values('odometer_categorie')


In [357]:
fig = px.bar(
    marge_odometer,
    x="marge_mediane_pct",
    y="odometer_categorie",
    orientation="h",
    title="Marge médiane en % par catégorie d'odometer",
    labels={
        "marge_mediane_pct": "Marge médiane (%)",
        "odometer_categorie": "Catégorie odometer"
    },
    color="marge_mediane_pct",
    color_continuous_scale=["red", "lightgrey", "steelblue"],
    color_continuous_midpoint=0   # ← rouge si négatif, bleu si positif
)

fig.add_vline(
    x=0,
    line_color="black",
    line_width=1
)

fig.update_layout(
    yaxis=dict(type="category"),
    coloraxis_showscale=False
)
fig.show()

On constate que l'odometer apparaît quasi-exclusivement comme quelque chose de négatif pour ce qui concerne la marge, sauf pour les odometers en 300 000 et 700 000.



Observons à l'aide d'un schéma comment le volume se comporte par rapport à l'odometer.


In [358]:
fig = px.bar(
    marge_odometer,
    x="volume",
    y="odometer_categorie",
    orientation="h",
    title="Volume de ventes par catégorie de odometer",
    labels={
        "volume": "Nombre de ventes",
        "odometer_categorie": "Catégorie odometer"
    },
    color="volume",
    color_continuous_scale=["lightgrey", "steelblue"],
)

fig.update_layout(
  yaxis=dict(type="category"),
    coloraxis_showscale=False

)
fig.show()

Les voitures avec peu d'odometer sont ceux qui se vendent le mieux (hormis ceux de moins de 10 000 km).Mais cela n'empèche pas les voitures ayant entre 70000 odometer et 125 000 odometer de bien se vendre.

Visualisons avec un shéma, si les catégories avec plus de ventes ont aussi des marges différentes.

In [359]:
fig = px.scatter(
    marge_odometer,
    x="odometer_categorie",
    y="marge_mediane_pct",
    size="volume",
    color="marge_mediane_pct",
    color_continuous_scale=["red", "lightgrey", "steelblue"],
    color_continuous_midpoint=0,   # ← rouge si marge négative, bleu si positive
    title="Volume de ventes vs Marge par condition",
    labels={
        "condition_label": "Condition",
        "marge_mediane_pct": "Marge médiane (%)",
        "volume": "Volume de ventes"
    },
    size_max=60
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
    line_dash="dash"
)

fig.update_layout(
    xaxis=dict(type="category"),
    coloraxis_showscale=False
)
fig.show()

Le shéma montre que lorsque le volume est faible la marge part plus dans les extrème (fortement positive ou fortement négative), et qu'après 100 000 km il y a moins de chance de faire des marges positives.

** Par age du véhicule lors de la vente

In [360]:
data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558709 entries, 0 to 558708
Data columns (total 27 columns):
 #   Column               Non-Null Count   Dtype   
---  ------               --------------   -----   
 0   level_0              558709 non-null  int64   
 1   index                558709 non-null  int64   
 2   year                 558709 non-null  int64   
 3   make                 548498 non-null  object  
 4   model                548400 non-null  object  
 5   trim                 548148 non-null  object  
 6   body                 545604 non-null  object  
 7   transmission         493376 non-null  object  
 8   vin                  558709 non-null  object  
 9   state                558709 non-null  object  
 10  condition            558709 non-null  float64 
 11  odometer             558709 non-null  float64 
 12  color                557962 non-null  object  
 13  interior             557962 non-null  object  
 14  seller               558709 non-null  object  
 15  

Je calcule l'age du véhicule au moment de sa vente.

In [361]:
data['age'] = data['year_s']-data['year']
data.head(10)

,level_0,index,year,make,model,trim,body,transmission,vin,state,...,condition_was_nan,odometer_was_nan,day,month,year_s,marge,marge_pct,condition_categorie,odometer_categorie,age
0,0,0,2015,kia,sorento,lx,suv,automatic,5xyktca69fg566472,ca,...,False,False,Tue,Dec,2014,1000.00,4.88,0,20000,-1
1,1,1,2015,kia,sorento,lx,suv,automatic,5xyktca69fg561319,ca,...,False,False,Tue,Dec,2014,700.00,3.37,0,10000,-1
2,2,2,2014,bmw,3 series,328i sulev,sedan,automatic,wba3c1c51ek116351,ca,...,False,False,Thu,Jan,2015,-1900.00,-5.96,4,10000,1
3,3,3,2015,volvo,s60,t5,sedan,automatic,yv1612tb4f1310987,ca,...,False,False,Thu,Jan,2015,250.00,0.91,4,20000,0
4,4,4,2014,bmw,6 series gran coupe,650i,sedan,automatic,wba6b2c57ed129731,ca,...,False,False,Thu,Dec,2014,1000.00,1.52,4,10000,0
5,5,5,2015,nissan,altima,2.5 s,sedan,automatic,1n4al3ap1fn326013,ca,...,False,False,Tue,Dec,2014,-4450.00,-28.99,0,10000,-1
6,6,6,2014,bmw,m5,base,sedan,automatic,wbsfv9c51ed593089,ca,...,False,False,Wed,Dec,2014,-4000.00,-5.80,3,20000,0
7,7,7,2014,chevrolet,cruze,1lt,sedan,automatic,1g1pc5sb2e7128460,ca,...,False,False,Tue,Dec,2014,-2100.00,-17.65,0,30000,0
8,8,8,2014,audi,a4,2.0t premium plus quattro,sedan,automatic,wauffafl3en030343,ca,...,False,False,Thu,Dec,2014,150.00,0.47,4,10000,0
9,9,9,2014,chevrolet,camaro,lt,convertible,automatic,2g1fb3d37e9218789,ca,...,False,False,Tue,Jan,2015,-8800.00,-33.46,0,10000,1


In [362]:
data['year'].describe()

,year
count,558709.00
mean,2010.04
std,3.97
min,1982.00
25%,2007.00
50%,2012.00
75%,2013.00
max,2015.00


In [363]:
data['year_s'].describe()

,year_s
count,558709.00
mean,2014.90
std,0.29
min,2014.00
25%,2015.00
50%,2015.00
75%,2015.00
max,2015.00


J'ai des age égalent à -1, est-ce un probleme de fuseau horaire. A-t-on bien à chaque fois une date de vente en décembre 2014 ?

In [364]:
data_age = data[data['age']==-1]

ok = data_age[
    (data_age['year'] == 2015) &
    (data_age['year_s'] == 2014) &
    (data_age['month'] == "Dec")
].shape[0] == data_age.shape[0]

print(ok)


False


In [365]:
data_bad = data[
    (data["age"] == -1) &
    ~(
        (data["year"] == 2015) &
        (data["year_s"] == 2014) &
        (data["month"] == "Dec")
    )
]

data_bad

,level_0,index,year,make,model,trim,body,transmission,vin,state,...,condition_was_nan,odometer_was_nan,day,month,year_s,marge,marge_pct,condition_categorie,odometer_categorie,age
60528,60528,60616,2015,hyundai,sonata,se,sedan,automatic,5npe24afxfh005731,ga,...,False,False,Mon,Jan,2014,1250.00,8.65,4,30000,-1
68845,68845,68933,2015,hyundai,sonata,se,sedan,automatic,5npe24af8fh005775,ga,...,False,False,Mon,Jan,2014,1050.00,7.17,4,20000,-1
68856,68856,68944,2015,hyundai,sonata,se,sedan,automatic,5npe24af8fh005761,ga,...,False,False,Mon,Jan,2014,1150.00,7.96,4,30000,-1
68859,68859,68947,2015,hyundai,sonata,se,sedan,automatic,5npe24af4fh005529,ga,...,False,False,Mon,Jan,2014,1400.00,9.86,4,30000,-1
68860,68860,68948,2015,kia,sorento,lx,suv,automatic,5xykt3a68fg566463,sc,...,False,False,Wed,Jan,2014,-450.00,-2.37,0,20000,-1


Non, j'ai donc des incohérences de date, je vais les garder et juste les exclure lorsque je travaille sur l'age de la voiture.
Combien d'incohérence de date ?

In [366]:
print("J'ai " , data[data['age']==-1].shape[0], 'dates incohérentes. Soit ', data[data['age']==-1].shape[0]/len(data)*100, '% de mes données')

J'ai  201 dates incohérentes. Soit  0.035975794196979105 % de mes données


On va regarder les marges en fonction de l'age de la voiture

In [367]:
data_ok = data[data['age']!=-1].copy()
marge_age=data_ok.groupby('age',as_index=False).agg(marge_mediane_pct=('marge_pct','median'), marge_mediane=('marge','median'), volume=('index', 'count'))
marge_age.sort_values('marge_mediane_pct', ascending=False, inplace=True)
marge_age

,age,marge_mediane_pct,marge_mediane,volume
33,33,219.40,7825.00,2
31,31,19.54,350.00,5
29,29,0.00,0.00,12
3,3,-0.25,-25.00,99850
4,4,-0.28,-50.00,46202
2,2,-0.29,-50.00,98969
5,5,-0.30,-25.00,26039
1,1,-0.35,-50.00,82640
6,6,-0.38,-50.00,21891
7,7,-0.41,-25.00,31743


Y a-t-il correlation entre l'âge du véhicule et la marge ?

In [368]:
pd.DataFrame(pearsonr(data_ok['marge_pct'], data_ok['age']), index=['pearson_coeff','p-value'], columns=['resultat_test'])

,resultat_test
pearson_coeff,0.03
p-value,0.00


La p-value est inférieur à 5% on rejette donc le fait que la marge en pourcentage et l'age soit indépendante

In [369]:
age_data = data_ok[['marge', 'marge_pct', 'age']].copy()

In [370]:
# Affichage de la heatmap de ces corrélations
age_data.corr().style.background_gradient(cmap='Reds')

,marge,marge_pct,age
marge,1.000000,0.375976,0.034074
marge_pct,0.375976,1.000000,0.027243
age,0.034074,0.027243,1.000000


Il n'y a pas de corrélation forte.

In [371]:
fig = px.bar(
    marge_age,
    x="marge_mediane_pct",
    y="age",
    orientation="h",
    title="Marge médiane en % par âge",
    labels={
        "marge_mediane_pct": "Marge médiane (%)",
        "age": "Âge du véhicule"
    },
    color="marge_mediane_pct",
    color_continuous_scale=["red", "lightgrey", "steelblue"],
    color_continuous_midpoint=0
)

fig.add_vline(x=0, line_color="black", line_width=1)
fig.update_layout(
    coloraxis_showscale=False
)
fig.show()

Quel est le lien entre l'âge de la voiture et le volume de ventes ?

In [372]:
data_ok = data[data['age']!=-1].copy()
marge_age=data_ok.groupby('age',as_index=False).agg(marge_mediane_pct=('marge_pct','median'), marge_mediane=('marge','median'), volume=('index', 'count'))
marge_age.sort_values('volume', ascending=False, inplace=True)
marge_age

,age,marge_mediane_pct,marge_mediane,volume
3,3,-0.25,-25.00,99850
2,2,-0.29,-50.00,98969
1,1,-0.35,-50.00,82640
4,4,-0.28,-50.00,46202
7,7,-0.41,-25.00,31743
8,8,-0.79,-50.00,30563
9,9,-1.39,-75.00,26319
5,5,-0.30,-25.00,26039
6,6,-0.38,-50.00,21891
10,10,-2.26,-100.00,20934


In [373]:
fig = px.bar(
    marge_age,
    x="age",
    y="volume",
    orientation="v",
    title="Volume de vente par âge",
    labels={
        "volume": "Volume de vente",
        "age": "Âge du véhicule"
    },
    color="volume",
    color_continuous_scale=["red", "lightgrey", "steelblue"],
    color_continuous_midpoint=0
)

fig.add_vline(x=0, line_color="black", line_width=1)
fig.update_layout(
    coloraxis_showscale=False
)
fig.show()

L'âge de la voiture va influer le volume de vente. Au dessus de 20 ans la voiture à peu ou pas de chance d'être vendue.
Les voitures les plus vendues ont entre 1 et 3 ans.

### 3.2 Prix, marge, volume en fonction de certains éléments catégorielles (body, mark, transmission, color)

* On commence par s'intéresser au body.

In [374]:
data_body = data.groupby("body", as_index=False, observed=True).agg(
    marge_mediane_pct=("marge_pct", "median"),
    volume=("index", "count")
)
data_body

,body,marge_mediane_pct,volume
0,access cab,0.68,294
1,beetle convertible,0.29,59
2,cab plus,-21.25,4
3,cab plus 4,10.81,6
4,club cab,3.30,178
5,convertible,-0.25,10476
6,coupe,-0.29,17752
7,crew cab,0.00,16394
8,crewmax cab,0.53,565
9,cts coupe,0.22,158


Le top 10 pour le volume de vente par rapport au body.

In [375]:
data_body.sort_values('volume', ascending=False, inplace=True)
data_body[['body', 'volume']].head(10)

,body,volume
36,sedan,241332
39,suv,143844
24,hatchback,26237
28,minivan,25529
6,coupe,17752
7,crew cab,16394
43,wagon,16128
5,convertible,10476
38,supercrew,9033
19,g sedan,7417


In [376]:
volume_sedan = data_body.loc[data_body['body'] == 'sedan', 'volume'].iloc[0]


volume_suv = data_body.loc[data_body['body'] == 'suv', 'volume'].iloc[0]


volume_total = data_body['volume'].sum()
print(volume_total)

volume_hors = volume_total - (volume_sedan + volume_suv)



print('le pourcentage de Sedan vendu', volume_sedan*100/volume_total)
print('le pourcentage de SUV vendu', volume_suv*100/volume_total)

545604
le pourcentage de Sedan vendu 44.23208040996767
le pourcentage de SUV vendu 26.364176215716892


C'esgt une part non négligeable du marché.

In [377]:
# Liste cible
wanted_order = ["Sedan", "SUV", "Autres"]

data_body_group = pd.DataFrame({"body_categorie": wanted_order})

data_body_group['volume']= [volume_sedan, volume_suv, volume_hors]

data_body_group

,body_categorie,volume
0,Sedan,241332
1,SUV,143844
2,Autres,160428


In [378]:
fig = px.pie(
    data_body_group,
    values="volume",
    names="body_categorie",
    title="Volume par Sedan, SUV, Autres"
)

fig.show()

Le Sedan et les SUV représente plus des trois quart du marché.

Le top 10 pour la marge médiane en pct par rapport au body

In [379]:
data_body.sort_values('marge_mediane_pct', ascending=False, inplace=True)
data_body[['body', 'marge_mediane_pct']].head(10)

,body,marge_mediane_pct
12,cts-v wagon,19.39
3,cab plus 4,10.81
10,cts wagon,5.18
41,tsx sport wagon,3.98
4,club cab,3.30
30,q60 convertible,2.16
22,genesis coupe,2.08
29,promaster cargo van,1.32
35,regular-cab,1.18
0,access cab,0.68


In [380]:
print('La marge médiane en pourcentage des SUV est ', data_body.loc[data_body['body'] == 'suv', 'marge_mediane_pct'].iloc[0])
print('La marge médiane en pourcentage des Sedan est ', data_body.loc[data_body['body'] == 'sedan', 'marge_mediane_pct'].iloc[0])

La marge médiane en pourcentage des SUV est  -0.3058103975535168
La marge médiane en pourcentage des Sedan est  -0.5076142131979695


Ce que l'on vend le plus en terme de body (Sedan et SUV) n'est pas ce qui rapportera la meilleure marge médiane. Plus, les SUV et le Sedan ne permettent pas vraiment de se faire une marge conséquente.

Lien entre le prix de vente et le body

In [381]:
model_body = ols("sellingprice ~ C(body)", data=data).fit()
anova_table_body = sm.stats.anova_lm(model_body, typ=2)
anova_table_body

,sum_sq,df,F,PR(>F)
C(body),5436684169921.65,44.00,1460.15,0.00
Residual,46166357268114.73,545559.00,NaN,NaN


Les modalités testées (le body) ont un effet statistique significatif sur le prix de vente.

* On s'intéresse à la marque

In [382]:
data_make = data.groupby("make", as_index=False, observed=True).agg(
    marge_mediane_pct=("marge_pct", "median"),
    volume=("index", "count")
)
data_make

,make,marge_mediane_pct,volume
0,acura,0.25,5926
1,airstream,140.68,1
2,aston martin,1.74,25
3,audi,0.00,5877
4,bentley,0.00,116
...,...,...,...
61,tesla,-1.39,23
62,toyota,-0.40,39966
63,volkswagen,-0.50,12579
64,volvo,-0.44,3788


Le top 10 du volume de vente par rapport à la marque

In [383]:
data_make.sort_values('volume', ascending=False, inplace=True)
data_make[['make', 'volume']].head(10)

,make,volume
18,ford,93996
9,chevrolet,60587
48,nissan,54017
62,toyota,39966
12,dodge,30953
24,honda,27351
26,hyundai,21831
5,bmw,20793
32,kia,18082
10,chrysler,17483


Le top 10 de marge médiane en % par rapport à la marque

In [384]:
data_make.sort_values('marge_mediane_pct', ascending=False, inplace=True)
data_make[['make', 'marge_mediane_pct']].head(10)

,make,marge_mediane_pct
1,airstream,140.68
41,mazda tk,20.93
27,hyundai tk,12.00
8,chev truck,3.90
25,hummer,2.26
23,gmc truck,1.96
2,aston martin,1.74
38,lotus,1.24
60,suzuki,0.90
0,acura,0.25


A part pour Nissan (qui apparaît dans les 2 top 10) la marque n'influe pas la marge et le volume de vente de la même façon.

Lien entre le prix de vente et la marque

In [385]:
model_make = ols("sellingprice ~ C(make)", data=data).fit()
anova_table_make = sm.stats.anova_lm(model_make, typ=2)
anova_table_make

,sum_sq,df,F,PR(>F)
C(make),10599781008434.89,65.00,2164.86,0.00
Residual,41312052749773.50,548432.00,NaN,NaN


Les modalités testées (la marque) ont un effet statistique significatif sur le prix de vente.


* on s'interesse à la transmission

In [386]:
data_transmission = data.groupby("transmission", as_index=False, observed=True).agg(
    marge_mediane_pct=("marge_pct", "median"),
    volume=("index", "count")
)
data_transmission

,transmission,marge_mediane_pct,volume
0,automatic,-0.35,475836
1,manual,-2.64,17540


Les voitures automatique ont une meilleur marge médiane, et un nettement meilleur volume de vente.

Lien entre le prix de vente et le type de transmission

In [387]:
model_transmission = ols("sellingprice ~ C(transmission)", data=data).fit()
anova_table_transmission = sm.stats.anova_lm(model_transmission, typ=2)
anova_table_transmission

,sum_sq,df,F,PR(>F)
C(transmission),91719485136.07,1.00,978.22,0.00
Residual,46259602528088.33,493374.00,NaN,NaN


Les modalités testées (la transmission) ont un effet statistique significatif sur le prix de vente.

* on s'interesse à la couleur du véhicule

In [388]:
data_color = data.groupby("color", as_index=False, observed=True).agg(
    marge_mediane_pct=("marge_pct", "median"),
    volume=("index", "count")
)
data_color

,color,marge_mediane_pct,volume
0,beige,-1.28,9219
1,black,-0.58,110953
2,blue,-1.20,51127
3,brown,-0.82,6715
4,burgundy,-0.65,8966
5,charcoal,-0.70,479
6,gold,-0.75,11342
7,gray,-0.70,82847
8,green,-2.31,11372
9,lime,3.33,15


In [389]:
data_color_plt = data_color.drop(data_color.index[19])
data_color_plt.sort_values('volume', inplace=True, ascending=False)

In [390]:
fig = px.histogram(
    data_color_plt,
    x="color",
    y="volume",
    title='Volume de ventes par couleur du véhicule',
    labels={
        "color": "couleur",
        "volume": "Nombre de ventes"
    },
)

fig.show()

Comme en France ce sont les voitures dont les couleurs sont le plus sobres qui se vendent le mieux : noir, blanc, argent ou gris.

In [391]:
data_color_plt.sort_values('marge_mediane_pct', inplace=True, ascending=False)

In [392]:
fig = px.histogram(
    data_color_plt,
    x="color",
    y='marge_mediane_pct',
    title='Marge médiane en pourcentage par couleur du véhicule',
    labels={
        "color": "couleur",
        'marge_mediane_pct': 'Marge médiane en pourcentage',
    },
)

fig.show()

En revanche, ce ne sont pas les couleurs les plus sobres qui permettent de faire une meilleure marge médiane.

Lien entre le prix de vente et la couleur du véhicule.

In [393]:
model_color = ols("sellingprice ~ C(color)", data=data).fit()
anova_table_color = sm.stats.anova_lm(model_color, typ=2)
anova_table_color

,sum_sq,df,F,PR(>F)
C(color),2404782170576.92,19.00,1401.81,0.00
Residual,50375928989724.92,557942.00,NaN,NaN


Les modalités testées (la couleur) ont un effet statistique significatif sur le prix de vente

Si on constate que la couleur et le type de transmission ont un lien direct avec le volume de vente, le prix, la médiane de la marge en pourcentage. L'odometer et la condition n'ont pas un impact aussi linéaire que l'on pouvait l'imaginer. On va donc voir ce qu'il en est lorsqu'on croise ces attributs avec le body de la voiture.

### 3.3. Valeurs croisées

#### 3.3.1 body x condition

In [394]:
data_body_condition = data.groupby(
    ['body', 'condition_categorie'],
    as_index=False, observed=True
).agg(
    marge_mediane_pct=('marge_pct', 'median'),
    volume=('index', 'count'),
    sellingprice_median=('sellingprice', 'median'),
    age_median=('age', 'median'),
    odometer_median=('odometer', 'median')
).copy()
data_body_condition



,body,condition_categorie,marge_mediane_pct,volume,sellingprice_median,age_median,odometer_median
0,access cab,0,-5.04,24,10725.00,9.00,81117.00
1,access cab,1,-19.16,32,4400.00,11.00,145521.50
2,access cab,2,-1.51,81,7800.00,10.00,114791.00
3,access cab,3,1.41,86,15250.00,4.00,73526.00
4,access cab,4,1.71,71,21000.00,2.00,28545.00
...,...,...,...,...,...,...,...
194,xtracab,0,-1.16,7,5000.00,12.00,179541.00
195,xtracab,1,-15.22,12,4500.00,12.50,141855.50
196,xtracab,2,-0.64,15,5900.00,13.00,167059.00
197,xtracab,3,8.14,8,6100.00,13.50,190358.00


Affichage par diagramme

In [395]:
fig = px.bar(
    data_body_condition,
    x="body",
    y="volume",
    color="condition_categorie",
    barmode="group",
    title="Volume de vente par body et condition_categorie",
    labels={"body": "body", "volume": "Volume", "condition_categorie": "Condition"}
)

fig.show()

Au vu de la suprématie des SUV et des Sedan. L'on va regrouper les body autres que Sedan et SUV dans une même catégorie : Autres.

In [396]:
def body_group(body_val):
    if body_val == "suv":
        return "SUV"
    elif body_val == "sedan":
        return "Sedan"
    else:
        return "Autres"


In [397]:
data_body_condition



,body,condition_categorie,marge_mediane_pct,volume,sellingprice_median,age_median,odometer_median
0,access cab,0,-5.04,24,10725.00,9.00,81117.00
1,access cab,1,-19.16,32,4400.00,11.00,145521.50
2,access cab,2,-1.51,81,7800.00,10.00,114791.00
3,access cab,3,1.41,86,15250.00,4.00,73526.00
4,access cab,4,1.71,71,21000.00,2.00,28545.00
...,...,...,...,...,...,...,...
194,xtracab,0,-1.16,7,5000.00,12.00,179541.00
195,xtracab,1,-15.22,12,4500.00,12.50,141855.50
196,xtracab,2,-0.64,15,5900.00,13.00,167059.00
197,xtracab,3,8.14,8,6100.00,13.50,190358.00


In [398]:
# ajout des catégories autres

data['body_categorie']= data['body'].apply(body_group)
data


,level_0,index,year,make,model,trim,body,transmission,vin,state,...,odometer_was_nan,day,month,year_s,marge,marge_pct,condition_categorie,odometer_categorie,age,body_categorie
0,0,0,2015,kia,sorento,lx,suv,automatic,5xyktca69fg566472,ca,...,False,Tue,Dec,2014,1000.00,4.88,0,20000,-1,SUV
1,1,1,2015,kia,sorento,lx,suv,automatic,5xyktca69fg561319,ca,...,False,Tue,Dec,2014,700.00,3.37,0,10000,-1,SUV
2,2,2,2014,bmw,3 series,328i sulev,sedan,automatic,wba3c1c51ek116351,ca,...,False,Thu,Jan,2015,-1900.00,-5.96,4,10000,1,Sedan
3,3,3,2015,volvo,s60,t5,sedan,automatic,yv1612tb4f1310987,ca,...,False,Thu,Jan,2015,250.00,0.91,4,20000,0,Sedan
4,4,4,2014,bmw,6 series gran coupe,650i,sedan,automatic,wba6b2c57ed129731,ca,...,False,Thu,Dec,2014,1000.00,1.52,4,10000,0,Sedan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
558704,558704,558832,2015,kia,k900,luxury,sedan,NaN,knalw4d4xf6019304,in,...,False,Thu,Jul,2015,-2300.00,-6.52,4,20000,0,Sedan
558705,558705,558833,2012,ram,2500,power wagon,crew cab,automatic,3c6td5et6cg112407,wa,...,False,Wed,Jul,2015,600.00,1.99,0,60000,3,Autres
558706,558706,558834,2012,bmw,x5,xdrive35d,suv,automatic,5uxzw0c58cl668465,ca,...,False,Wed,Jul,2015,4200.00,14.09,4,60000,3,SUV
558707,558707,558835,2015,nissan,altima,2.5 s,sedan,automatic,1n4al3ap0fc216050,ga,...,False,Thu,Jul,2015,-4000.00,-26.49,3,20000,0,Sedan


On regroupe pas categorie de body et par condition.

In [399]:

data_body_analyse = data.groupby(
    ['body_categorie', 'condition_categorie'],
    as_index=False, observed=True,
).agg(
    marge_mediane_pct=('marge_pct', 'median'),
    volume=('index', 'count'),
    sellingprice_median=('sellingprice', 'median'),
    age_median=('age', 'median'),
    odometer_median=('odometer', 'median')
).copy()




In [400]:

data_body_analyse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   body_categorie       15 non-null     object 
 1   condition_categorie  15 non-null     int64  
 2   marge_mediane_pct    15 non-null     float64
 3   volume               15 non-null     int64  
 4   sellingprice_median  15 non-null     float64
 5   age_median           15 non-null     float64
 6   odometer_median      15 non-null     float64
dtypes: float64(4), int64(2), object(1)
memory usage: 972.0+ bytes


In [401]:
data_body_analyse_pivot = data_body_analyse.pivot(
    index='condition_categorie',
    columns='body_categorie',
    values=['marge_mediane_pct', 'volume','sellingprice_median','age_median', 'odometer_median']
).fillna(0)

data_body_analyse_pivot

marge_mediane_pct                 volume           \
body_categorie                 Autres    SUV  Sedan   Autres      SUV   
condition_categorie                                                     
0                               -1.89  -1.18  -3.23 23453.00 17198.00   
1                              -15.79 -17.06 -16.72 15885.00  7914.00   
2                               -3.96  -5.14  -4.92 38462.00 27767.00   
3                                0.42   0.00   0.31 48258.00 42269.00   
4                                1.80   1.48   2.14 47475.00 48696.00   

                             sellingprice_median                   age_median  \
body_categorie         Sedan              Autres      SUV    Sedan     Autres   
condition_categorie                                                             
0                   29764.00            12400.00 15900.00 10000.00       3.00   
1                   21815.00             3500.00  3700.00  3200.00       9.00   
2                   51777.00             7600.00  7600.00  7800.00       7.00   
3                   73809.00            13300.00 14400.00 11500.00       4.00   
4                   64167.00            19500.00 19900.00 14200.00       2.00   

                                odometer_median                      
body_categorie        SUV Sedan          Autres       SUV     Sedan  
condition_categorie                                                  
0                    3.00  3.00        44167.00  43147.00  42275.50  
1                   10.00  9.00       122846.00 126778.50 113351.00  
2                    8.00  5.00        95623.50 103633.00  77922.00  
3                    4.00  3.00        56903.50  65536.00  46806.00  
4                    2.00  2.00        28548.00  35049.50  29121.00

On va maintenant visualiser.

Quelle body_categorie et condition_categorie ont la meilleure marge ?

In [402]:
fig = px.density_heatmap(
    data_body_analyse,
    x="body_categorie",
    y="condition_categorie",
    z="marge_mediane_pct",

    histfunc="avg",
    color_continuous_scale="RdYlGn",
    title="Marge médiane (%) par body × condition",
    labels={"marge_mediane_pct": "Marge (%)"}
)
fig.update_layout(yaxis=dict(type="category"))
fig.show()

Les meilleures marges sont faites par les Sedan en très bon état (catégorie 4), et le groupement des autre catégorie de body.
La catégorie 4 est pour chaque type de body celle qui fait la plus haute marge

Comment chaque body réagit selon la condition pour la marge.

In [403]:
fig = px.bar(
    data_body_analyse,
    x="body_categorie",
    y="marge_mediane_pct",
    color="condition_categorie",
    text="volume",
    barmode="group",
    title="Marge médiane par body et condition",
    labels={
        "marge_mediane_pct": "Marge médiane (%)",
        "condition_categorie": "Condition",
        "body_categorie": "Body",
        "volume" : "Volume"
    }
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

Les voitures en bonne condition sont celles qui ont une marge positive. Néanmoins l'on peut noter que les voiture dans le plus mauvaise état ont une faible décote par rapport à la condition juste au-dessus.

Comment chaque body réagit selon la condition pour le volume de vente.

In [404]:
fig = px.bar(
    data_body_analyse,
    x="body_categorie",
    y="volume",
    color="condition_categorie",
    barmode="group",
    title="Volume par body et condition",
    labels={
        "volume": "Volume",
        "condition_categorie": "Condition",
        "body_categorie": "Body"
    }
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

Les Sedan sont les voitures qui se vende le plus.
On constate que les volumes sont corrélé à la condition quelque soit le type de body. Meilleure est la condition, meilleure est le volume.

Comment chaque body réagit selon la condition pour le prix de vente

In [405]:
fig = px.bar(
    data_body_analyse,
    x="body_categorie",
    y="sellingprice_median",
    color="condition_categorie",
    text="volume",
    barmode="group",
    title="prix médian par body et condition",
    labels={
        "sellingprice_median": "Prix de vente médian",
        "condition_categorie": "Condition",
        "body_categorie": "Body",
        "volume":"Volume"
    }
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

On constate que le prix médian et corrélé à la condition quelque soit le type de body. Meilleure est la condition, meilleure est le volume. Et que ce ne sont pas les voitures qui se vendent le plus (SUV, Sedan) qui se vendent le plus cher?

Arbitrage prix / marge / volume par segment.

In [406]:
fig = px.scatter(
    data_body_analyse,
    x="sellingprice_median",
    y="marge_mediane_pct",
    size="volume",
    color="condition_categorie",
    text="body_categorie",
    title="Prix vs Marge, taille = Volume, couleur = Condition",
    labels={
        "sellingprice_median": "Prix médian ($)",
        "marge_mediane_pct": "Marge médiane (%)",
        "condition_categorie": "Condition"
    },
    size_max=60
)
fig.update_traces(textposition="top center")
fig.show()

Le prix médian est le plus élevé pour les voitures en meilleur état, même sur les SUV en très mauvais état ont aussi un prix médian élevé. Encore une fois se sont les voitures dont l'état est 1 (donc celle juste au-dessus de la pire catégorie) qui ont le pire prix médian, une marge médiane très mauvaise et un volume de vente plus faible.

Une vision d'ensemble.

In [407]:
data_body_analyse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   body_categorie       15 non-null     object 
 1   condition_categorie  15 non-null     int64  
 2   marge_mediane_pct    15 non-null     float64
 3   volume               15 non-null     int64  
 4   sellingprice_median  15 non-null     float64
 5   age_median           15 non-null     float64
 6   odometer_median      15 non-null     float64
dtypes: float64(4), int64(2), object(1)
memory usage: 972.0+ bytes


In [408]:
body_codes = data_body_analyse["body_categorie"].astype("category").cat.codes


body_labels = data_body_analyse["body_categorie"].astype("category").cat.categories.tolist()


fig = go.Figure(data=go.Parcoords(
    line=dict(
        color=data_body_analyse["condition_categorie"],
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="Condition")
    ),
    dimensions=[
        dict(
            label="Body",
            values=body_codes,
            tickvals=list(range(len(body_labels))),
            ticktext=body_labels
        ),
        dict(label="Marge (%)", values=data_body_analyse["marge_mediane_pct"]),
        dict(label="Prix médian", values=data_body_analyse["sellingprice_median"]),
        dict(label="Volume", values=data_body_analyse["volume"]),
        dict(label="Âge médian", values=data_body_analyse["age_median"]),
        dict(label="Km médian", values=data_body_analyse["odometer_median"])
    ]
))
fig.update_layout(title="Parallel coordinates — profils body × condition")
fig.show()


#### 3.3.1 body x odometer

In [409]:
data_body_odometer = data.groupby(
    ['body', 'odometer_categorie'],
    as_index=False, observed=True
).agg(
    marge_mediane_pct=('marge_pct', 'median'),
    volume=('index', 'count'),
    sellingprice_median=('sellingprice', 'median'),
    age_median=('age', 'median'),

).copy()




Affichage par diagramme.

In [410]:
fig = px.bar(
    data_body_odometer,
    x="body",
    y="volume",
    color="odometer_categorie",
    barmode="group",
    title="Volume de vente par body et odometer_categorie",
    labels={"body": "body", "volume": "Volume", "odometer_categorie": "Odometer"}
)

fig.show()

Encore une fois les SUV et les Sedan écrasent le marché, on va donc faire 3 catégorie de body.

On regroupe pas type de body et catégorie d'odometer

In [411]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558709 entries, 0 to 558708
Data columns (total 29 columns):
 #   Column               Non-Null Count   Dtype   
---  ------               --------------   -----   
 0   level_0              558709 non-null  int64   
 1   index                558709 non-null  int64   
 2   year                 558709 non-null  int64   
 3   make                 548498 non-null  object  
 4   model                548400 non-null  object  
 5   trim                 548148 non-null  object  
 6   body                 545604 non-null  object  
 7   transmission         493376 non-null  object  
 8   vin                  558709 non-null  object  
 9   state                558709 non-null  object  
 10  condition            558709 non-null  float64 
 11  odometer             558709 non-null  float64 
 12  color                557962 non-null  object  
 13  interior             557962 non-null  object  
 14  seller               558709 non-null  object  
 15  

In [412]:
data_body_odo_analyse = data.groupby(
    ['body_categorie', 'odometer_categorie'],
    as_index=False, observed=True,
).agg(
    marge_mediane_pct=('marge_pct', 'median'),
    volume=('index', 'count'),
    sellingprice_median=('sellingprice', 'median'),
    age_median=('age', 'median'),
).copy()
data_body_odo_analyse.head()



,body_categorie,odometer_categorie,marge_mediane_pct,volume,sellingprice_median,age_median
0,Autres,10000,0.00,9534,20800.00,1.00
1,Autres,20000,0.00,19136,20500.00,1.00
2,Autres,30000,-0.23,19402,19600.00,2.00
3,Autres,40000,0.00,18163,17900.00,3.00
4,Autres,50000,0.42,12934,16600.00,3.00


On va maintenant visualiser.

Quel body_categorie et odometer_categorie fait les meilleures marges ?

In [413]:
fig = px.density_heatmap(
    data_body_odo_analyse,
    x="body_categorie",
    y="odometer_categorie",
    z="marge_mediane_pct",
    histfunc="avg",
    color_continuous_scale="RdYlGn",
    title="Marge médiane (%) par body × odometer",
    labels={"marge_mediane_pct": "Marge (%)"}
)
fig.update_layout(yaxis=dict(type="category"))
fig.show()

Plus les catégories au kilométrage le plus élevé font les meilleures marges pour les Sedan et les pires marges pour les autres catégories.
Ceci est à nuancer avec le nombre de vente pour ces catégories (faible).

Comment chaque body réagit selon l'odometer pour la marge.

In [414]:
fig = px.bar(
    data_body_odo_analyse,
    x="body_categorie",
    y="marge_mediane_pct",
    color="odometer_categorie",
    barmode="group",
    text="volume",                  # ← affiche le volume sur chaque barre
    title="Marge médiane par body et odometer",
    labels={
        "marge_mediane_pct": "Marge médiane (%)",
        "odometer_categorie": "Odometer",
        "body_categorie": "Body",
        "volume" : "Volume"
    }
)
fig.update_traces(textposition="outside")   # ← "inside" si les barres sont grandes
fig.update_layout(xaxis_tickangle=-30)
fig.show()

Rare sont les marges positives et encore une fois, elle ne concerne qu'un petit volume de vente.

Comment chaque body réagit selon l'odometer pour le volume de vente ?

In [415]:
fig = px.bar(
    data_body_odo_analyse,
    x="body_categorie",
    y="volume",
    color="odometer_categorie",
    barmode="group",
    title="Volume de vente par body et odometer",
    labels={
        "odometer_categorie": "Odometer",
        "body_categorie": "Body",
        "volume" : "Volume"
    }
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

L'odometer semble impacté le volume de vente de la même façon quel que soit le body. En revanche, le volume est toujours plus élevé lorsqu'il s'agit d'une Sedan.

Comment chaque body X odometer réagi pour le prix de ventes ?

In [416]:
fig = px.bar(
    data_body_odo_analyse,
    x="body_categorie",
    y="sellingprice_median",
    color="odometer_categorie",
    barmode="group",
    text="volume",                  # ← affiche le volume sur chaque barre
    title="Marge médiane par body et odometer",
    labels={
       "sellingprice_median": "Prix de vente médian",
        "odometer_categorie": "Odometer",
        "body_categorie": "Body",
        "volume" : "Volume"
    }
)
fig.update_traces(textposition="outside")   # ← "inside" si les barres sont grandes
fig.update_layout(xaxis_tickangle=-30)
fig.show()

 Tant que la voiture a un odometer inférieur à 300 000, moins il y a d'odometer, plus le prix de vente médian est élevé.

Prix/marge/volume par segment

In [417]:
fig = px.scatter(
    data_body_odo_analyse,
    x="sellingprice_median",
    y="marge_mediane_pct",
    size="volume",
    color="odometer_categorie",
    text="body_categorie",
    title="Prix vs Marge, taille = Volume, couleur = Odometer",
    labels={
        "sellingprice_median": "Prix médian ($)",
        "marge_mediane_pct": "Marge médiane (%)",
        "odometer_categorie": "Odometer"
    },
    size_max=60
)
fig.update_traces(textposition="top center")
fig.show()

Les SUV qui ont le moins d'odometer sont ceux qui ont un prix médian le plus élevé.
Le plus fort volume de ventes concernent les Sedan avec un odometer compris entre 30 000 et 40 000.

Une vision d'ensemble :

In [418]:
data_body_odo_analyse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   body_categorie       45 non-null     object  
 1   odometer_categorie   45 non-null     category
 2   marge_mediane_pct    45 non-null     float64 
 3   volume               45 non-null     int64   
 4   sellingprice_median  45 non-null     float64 
 5   age_median           45 non-null     float64 
dtypes: category(1), float64(3), int64(1), object(1)
memory usage: 2.6+ KB


In [419]:
fig = px.density_heatmap(
    data_body_odo_analyse,
    x="body_categorie",
    y="odometer_categorie",
    z="marge_mediane_pct",
    histfunc="avg",
    color_continuous_scale=["red", "lightgrey", "steelblue"],
    color_continuous_midpoint=0,
    title="Marge médiane (%) par body × odometer",
    labels={
        "body_categorie": "Body",
        "odometer_categorie": "Odometer",
        "marge_mediane_pct": "Marge (%)"
    }


)
fig.update_layout(
    yaxis=dict(type="category", categoryorder="category ascending"),
    xaxis=dict(tickangle=-30)
)
fig.show()

### 3.4 Une vision en fonction de la date

On va étudier le volume des ventes en fonction des mois de l'année, est des années

In [420]:
pd.options.display.float_format = '{:.2f}'.format
data_month=data.groupby(['month','year_s'], as_index=False, observed=True ).agg(
    volume=('index', 'count'),
    CA=('sellingprice', 'sum'),

  ).copy()
data_month

,month,year_s,volume,CA
0,Apr,2015,1450,14799755.00
1,Dec,2014,53433,604179255.00
2,Feb,2014,1,10500.00
3,Feb,2015,163052,2218852327.00
4,Jan,2014,206,3204525.00
5,Jan,2015,140608,1868474067.00
6,Jul,2015,1300,22069764.00
7,Jun,2015,99936,1499879515.00
8,Mar,2015,46276,622100423.00
9,May,2015,52447,752178456.00


In [421]:
mois_ordre = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}

data_month["month_num"] = data_month["month"].map(mois_ordre)

data_month = data_month.sort_values(["year_s", "month_num"])

# Label lisible pour les abscisse
data_month["mois_label"] = data_month["month"] + " " + data_month["year_s"].astype(str)

fig = make_subplots(rows = 2, cols = 1)


fig.add_trace(

        go.Scatter(

            x = data_month["mois_label"],
            y = data_month["volume"],
            mode = "markers+lines",
            name="Volume de ventes par mois",
        ),

        row = 1, col = 1
    )

fig.add_trace(
            go.Scatter(
            x = data_month["mois_label"],
            y = data_month["CA"],
            mode = "markers+lines",
            name="CA par mois",
        ),

        row = 2, col = 1
    )





fig.update_layout(
    xaxis1=dict(
        tickangle=-45,
        categoryorder="array",
        categoryarray=data_month["mois_label"].tolist()  # ← respecte l'ordre calendaire
    ),
    xaxis2=dict(
        tickangle=-45,
        categoryorder="array",
        categoryarray=data_month["mois_label"].tolist()  # ← respecte l'ordre calendaire
    )
)
fig.show()

On constate qu'il nous manque des données sur l'année 2014 pour savoir si les ventes sont cycliques sur une année.
De janvier 2014 à Julliet 2015, les meilleurs volumes de vente ont eu lieu en janvier et février 2015, mais en avril 2015 les ventes retombent. Malgré un rebond en juin 2015, elles retomberont à peu près au niveau d'avril 2015 en juillet 2015.

Le chiffre d'affaire suit les mêmes tendances que le volume des ventes.

### 3.5 une vision par état

On va étudier le volume de vente pour chaque état, ainsi que leur marge moyennne

In [422]:
# Dictionnaire code état et nom complet
state_names = {
    "al": "Alabama",
    "ak": "Alaska",
    "az": "Arizona",
    "ar": "Arkansas",
    "ca": "California",
    "co": "Colorado",
    "ct": "Connecticut",
    "de": "Delaware",
    "fl": "Florida",
    "ga": "Georgia",
    "hi": "Hawaii",
    "id": "Idaho",
    "il": "Illinois",
    "in": "Indiana",
    "ia": "Iowa",
    "ks": "Kansas",
    "ky": "Kentucky",
    "la": "Louisiana",
    "me": "Maine",
    "md": "Maryland",
    "ma": "Massachusetts",
    "mi": "Michigan",
    "mn": "Minnesota",
    "ms": "Mississippi",
    "mo": "Missouri",
    "mt": "Montana",
    "ne": "Nebraska",
    "nv": "Nevada",
    "nh": "New Hampshire",
    "nj": "New Jersey",
    "nm": "New Mexico",
    "ny": "New York",
    "nc": "North Carolina",
    "nd": "North Dakota",
    "oh": "Ohio",
    "ok": "Oklahoma",
    "or": "Oregon",
    "pa": "Pennsylvania",
    "ri": "Rhode Island",
    "sc": "South Carolina",
    "sd": "South Dakota",
    "tn": "Tennessee",
    "tx": "Texas",
    "ut": "Utah",
    "vt": "Vermont",
    "va": "Virginia",
    "wa": "Washington",
    "wv": "West Virginia",
    "wi": "Wisconsin",
    "wy": "Wyoming"
}

In [423]:
data_state=data.groupby('state', as_index=False, observed=True ).agg(
    volume=('index', 'count'),
    marge_mediane_pct=('marge_pct', 'median')
  ).copy()
# Ajouter la colonne avec le nom complet
data_state['state_name'] = data_state['state'].map(state_names)
data_state

,state,volume,marge_mediane_pct,state_name
0,ab,928,-2.01,NaN
1,al,26,-3.68,Alabama
2,az,8736,-0.49,Arizona
3,ca,73140,1.12,California
4,co,7775,0.37,Colorado
5,fl,82937,-0.46,Florida
6,ga,34749,0.51,Georgia
7,hi,1237,-5.47,Hawaii
8,il,23459,-0.60,Illinois
9,in,4325,-1.86,Indiana


In [424]:
fig = px.bar(
    data_state,
    x="state_name",
    y="volume",
    color="marge_mediane_pct",
    barmode="group",
    title="Volume de vente par état",
    labels={
        "state_name": "etat",
        "volume" : "Volume"
    }
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

La Californie, la floride et dans une moindre mesure la Pennsylvanie et le Texas tirent le marché vers le haut avec de fort volumes de vente.

On constate que les états où la marge médiane est vraiment mauvaise sont limités. Il s'agit de Hawai, du Massachussett, du Maryland et de New York.

## 4. Conclusions et limites
### Principaux enseignements :
*	Pour maximiser les chances de vente, il vaut mieux proposer une berline ou un SUV - mais ces carrosseries n'offrent pas la meilleure marge médiane.
*	L'âge du véhicule a un impact fort sur le volume : il est préférable de revendre un véhicule âgé de 1 à 3 ans.
*	Les couleurs sobres (noir, blanc, argent, gris) génèrent un meilleur volume de vente.
*	Les véhicules a transmission automatique affichent une meilleure marge médiane et un volume nettement supérieur.
*	Les véhicules en bonne condition ont une marge positive. Les véhicules en très mauvais état (condition 0) se comportent mieux que ceux de condition 1, tant en volume qu'en marge. Il convient donc d'éviter les véhicules de condition 1.
*	La Californie, la Floride, la Pennsylvanie et le Texas sont les marches les plus dynamiques.

### Limites de l'analyse :
*	La période couverte est d'environ un an et demi, avec des mois manquants : les analyses temporelles sont donc peu fiables.*
*	Seules les ventes effectivement réalisées sont connues. L'absence de données sur l'offre disponible au moment de chaque vente biaise toutes les analyses. Par exemple, est-ce que les voitures noires se vendent le plus parce que les acheteurs les préfèrent, ou simplement parce qu'elles représentaient la majorité de l'offre à qualité équivalente ?
